# EXP002 Objective Evaluation

### Inference Policy

HT-Demucs inference is conditionally executed by this notebook.

Before inference, the notebook checks every MUSDB18 test track for the four required prediction files:

- `vocals.wav`
- `drums.wav`
- `bass.wav`
- `other.wav`

If all 50 tracks have all four prediction files, inference is skipped completely.

If one or more tracks are incomplete, HT-Demucs inference is executed only for those incomplete tracks. Existing complete predictions are preserved.

This makes the notebook resumable and prevents unnecessary re-computation.

### Evaluation Location

The persistent EXP002 evaluation directory is:

`G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002`

### Important Runtime Note

This notebook is executed through Google Colab's **local runtime connection**.

The computation therefore runs in the local `iks_eval` Python environment on the Windows machine. Google Drive Desktop exposes the project files through the local `G:` filesystem path.

No `google.colab.drive.mount()` operation is required or should be used.

In [1]:
# ============================================================
# CELL 2: LOCAL RUNTIME & GPU VERIFICATION
# ============================================================

import sys
import os
import torch

print("=" * 70)
print("LOCAL RUNTIME VERIFICATION")
print("=" * 70)

# Python
print("\nPython executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

# Working directory
print("\nWorking directory:")
print(os.getcwd())

# PyTorch
print("\nPyTorch:")
print(torch.__version__)

# CUDA
print("\nCUDA available:")
print(torch.cuda.is_available())

print("\nTorch CUDA version:")
print(torch.version.cuda)

if torch.cuda.is_available():
    print("\nGPU:")
    print(torch.cuda.get_device_name(0))

    print("\nGPU count:")
    print(torch.cuda.device_count())

    print("\nGPU memory:")
    gpu_properties = torch.cuda.get_device_properties(0)
    print(f"Total VRAM: {gpu_properties.total_memory / (1024**3):.2f} GB")
else:
    print("\nWARNING: CUDA is NOT available.")

print("\n" + "=" * 70)

# Hard verification
expected_env = os.path.normcase(
    r"C:\IKS_Local_Runtime\iks_eval\Scripts\python.exe"
)

actual_env = os.path.normcase(sys.executable)

if actual_env == expected_env:
    print("✓ Correct local iks_eval environment detected.")
else:
    print("⚠ WARNING: Python executable does not match iks_eval.")
    print(f"Expected: {expected_env}")
    print(f"Actual:   {actual_env}")

if torch.cuda.is_available():
    print("✓ NVIDIA CUDA GPU detected.")
else:
    print("⚠ GPU acceleration unavailable.")

print("=" * 70)

LOCAL RUNTIME VERIFICATION

Python executable:
C:\IKS_Local_Runtime\iks_eval\Scripts\python.exe

Python version:
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]

Working directory:
C:\IKS_Local_Runtime

PyTorch:
2.11.0+cu128

CUDA available:
True

Torch CUDA version:
12.8

GPU:
NVIDIA GeForce RTX 3050 Laptop GPU

GPU count:
1

GPU memory:
Total VRAM: 4.00 GB

✓ Correct local iks_eval environment detected.
✓ NVIDIA CUDA GPU detected.


## Cell 3: EXP002 Project Configuration

This cell defines the local paths required for the EXP002 objective evaluation workflow.

The notebook is running through the local `iks_eval` Jupyter kernel, so Google Drive is accessed through the Windows-mounted Google Drive filesystem rather than `google.colab.drive`.

The existing EXP002 directory structure will be preserved.

### EXP002 structure

- `predictions/` → HT-Demucs separated outputs
- `metrics/` → numerical evaluation results
- `plots/` → evaluation plots and visualizations
- `reports/` → generated evaluation reports
- `logs/` → execution and evaluation logs
- `notebook/` → notebook files

The notebook will not recreate or reorganize these directories.

In [3]:
# ============================================================
# CELL 3: EXP002 PROJECT PATHS AND CONFIGURATION
# ============================================================

from pathlib import Path
import os

# ------------------------------------------------------------
# Google Drive / EXP002 root
# ------------------------------------------------------------

EXP002_ROOT = Path(
    r"G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation"
    r"\05_Experiments\Evaluation\Objective\EXP002"
)

# ------------------------------------------------------------
# EXP002 subdirectories
# ------------------------------------------------------------

PREDICTIONS_DIR = EXP002_ROOT / "predictions"
METRICS_DIR = EXP002_ROOT / "metrics"
PLOTS_DIR = EXP002_ROOT / "plots"
REPORTS_DIR = EXP002_ROOT / "reports"
LOGS_DIR = EXP002_ROOT / "logs"
NOTEBOOK_DIR = EXP002_ROOT / "notebook"

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("=" * 70)
print("EXP002 PROJECT CONFIGURATION")
print("=" * 70)

print(f"\nEXP002 root:")
print(EXP002_ROOT)

print("\nConfigured directories:")

paths = {
    "Predictions": PREDICTIONS_DIR,
    "Metrics": METRICS_DIR,
    "Plots": PLOTS_DIR,
    "Reports": REPORTS_DIR,
    "Logs": LOGS_DIR,
    "Notebook": NOTEBOOK_DIR,
}

for name, path in paths.items():
    print(f"  {name:<12}: {path}")

print("\n" + "=" * 70)
print("PATH EXISTENCE CHECK")
print("=" * 70)

for name, path in paths.items():
    status = "✓ EXISTS" if path.exists() else "✗ MISSING"
    print(f"{status:<10} {name}: {path}")

print("\n" + "=" * 70)

if not EXP002_ROOT.exists():
    raise FileNotFoundError(
        f"EXP002 root directory was not found:\n{EXP002_ROOT}\n\n"
        "Check the Google Drive Desktop path before continuing."
    )

print("✓ EXP002 root directory detected.")
print("✓ Configuration loaded successfully.")

EXP002 PROJECT CONFIGURATION

EXP002 root:
G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002

Configured directories:
  Predictions : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\predictions
  Metrics     : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics
  Plots       : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\plots
  Reports     : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports
  Logs        : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\logs
  Notebook    : G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\notebook

PATH EXISTENCE CHECK
✓ EXISTS  

## Cell 4: MUSDB18 Test-Set Discovery and Verification

Before running or skipping HT-Demucs inference, the notebook must establish the exact location of the MUSDB18 test set used by EXP002.

The MUSDB18 test set is expected to contain 50 tracks. Each track provides the four reference sources required for objective source-separation evaluation:

- `vocals`
- `drums`
- `bass`
- `other`

This cell searches the known local project locations for the MUSDB18 test directory and verifies that the expected 50-track test set is available.

No inference or evaluation is performed in this cell.

In [4]:
# ============================================================
# CELL 4: MUSDB18 TEST-SET DISCOVERY AND VERIFICATION
# ============================================================

from pathlib import Path

print("=" * 70)
print("MUSDB18 TEST-SET DISCOVERY")
print("=" * 70)

# ------------------------------------------------------------
# Candidate locations
# ------------------------------------------------------------
# We check the locations most likely to contain the existing
# MUSDB18 dataset used by the EXP002 workflow.
#
# These are discovery locations only. Nothing is moved or copied.
# ------------------------------------------------------------

candidate_roots = [
    Path(r"G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation"),
    Path(r"G:\My Drive"),
    Path(r"C:\IKS_Local_Runtime"),
]

candidate_test_dirs = []

print("\nSearching for MUSDB18 test directories...\n")

for root in candidate_roots:

    if not root.exists():
        print(f"⚠ Root does not exist: {root}")
        continue

    print(f"Checking: {root}")

    try:
        for path in root.rglob("test"):
            if path.is_dir():

                # A genuine MUSDB18 test directory should contain
                # track directories rather than ordinary files.
                try:
                    subdirs = [
                        p for p in path.iterdir()
                        if p.is_dir()
                    ]
                except PermissionError:
                    continue

                if len(subdirs) >= 40:
                    candidate_test_dirs.append(path)

    except (PermissionError, OSError) as exc:
        print(f"  Search warning: {exc}")

# ------------------------------------------------------------
# Remove duplicate paths
# ------------------------------------------------------------

candidate_test_dirs = list(dict.fromkeys(candidate_test_dirs))

print("\n" + "=" * 70)
print("CANDIDATE MUSDB18 TEST DIRECTORIES")
print("=" * 70)

if not candidate_test_dirs:
    print("\n✗ No likely MUSDB18 test directory was found.")
    print("\nWe will NOT continue to inference/evaluation yet.")
    print("The MUSDB18 location must be established first.")
    raise FileNotFoundError(
        "MUSDB18 test directory could not be located automatically."
    )

for i, path in enumerate(candidate_test_dirs, start=1):
    print(f"\n[{i}] {path}")

# ------------------------------------------------------------
# Inspect candidates
# ------------------------------------------------------------

valid_candidates = []

print("\n" + "=" * 70)
print("MUSDB18 TEST-SET VALIDATION")
print("=" * 70)

for test_dir in candidate_test_dirs:

    track_dirs = sorted([
        p for p in test_dir.iterdir()
        if p.is_dir()
    ])

    valid_track_count = 0

    for track_dir in track_dirs:

        required_stems = {
            "mixture.wav",
            "vocals.wav",
            "drums.wav",
            "bass.wav",
            "other.wav",
        }

        available_files = {
            p.name.lower()
            for p in track_dir.iterdir()
            if p.is_file()
        }

        if required_stems.issubset(available_files):
            valid_track_count += 1

    print(f"\nDirectory:")
    print(f"  {test_dir}")

    print(f"Track directories found: {len(track_dirs)}")
    print(f"Tracks with all 5 required WAV files: {valid_track_count}")

    if valid_track_count == 50:
        valid_candidates.append(test_dir)
        print("✓ Valid 50-track MUSDB18 test set detected.")
    else:
        print("⚠ Does not contain exactly 50 fully valid test tracks.")

# ------------------------------------------------------------
# Final selection
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MUSDB18 TEST-SET RESULT")
print("=" * 70)

if len(valid_candidates) == 0:
    raise RuntimeError(
        "No candidate containing exactly 50 valid MUSDB18 test tracks "
        "was found. Do not proceed until the dataset location is verified."
    )

if len(valid_candidates) > 1:
    print("\n⚠ Multiple valid MUSDB18 test directories were found:")
    for path in valid_candidates:
        print(f"  {path}")

    raise RuntimeError(
        "Multiple valid MUSDB18 test directories were found. "
        "We need to identify the exact dataset used by EXP002 before continuing."
    )

MUSDB18_TEST_DIR = valid_candidates[0]

print(f"\n✓ MUSDB18_TEST_DIR:")
print(f"  {MUSDB18_TEST_DIR}")

# ------------------------------------------------------------
# Store the 50 track names
# ------------------------------------------------------------

MUSDB18_TRACK_DIRS = sorted([
    p for p in MUSDB18_TEST_DIR.iterdir()
    if p.is_dir()
])

MUSDB18_TRACK_NAMES = [
    p.name for p in MUSDB18_TRACK_DIRS
]

print(f"\n✓ Number of test tracks: {len(MUSDB18_TRACK_NAMES)}")

print("\nFirst 5 tracks:")
for name in MUSDB18_TRACK_NAMES[:5]:
    print(f"  - {name}")

print("\nLast 5 tracks:")
for name in MUSDB18_TRACK_NAMES[-5:]:
    print(f"  - {name}")

print("\n" + "=" * 70)
print("✓ MUSDB18 TEST-SET VERIFIED")
print("=" * 70)

MUSDB18 TEST-SET DISCOVERY

Searching for MUSDB18 test directories...

Checking: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation
Checking: G:\My Drive
Checking: C:\IKS_Local_Runtime

CANDIDATE MUSDB18 TEST DIRECTORIES

[1] G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\03_Datasets\Benchmarks\MUSDB18\test

MUSDB18 TEST-SET VALIDATION

Directory:
  G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\03_Datasets\Benchmarks\MUSDB18\test
Track directories found: 50
Tracks with all 5 required WAV files: 50
✓ Valid 50-track MUSDB18 test set detected.

FINAL MUSDB18 TEST-SET RESULT

✓ MUSDB18_TEST_DIR:
  G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\03_Datasets\Benchmarks\MUSDB18\test

✓ Number of test tracks: 50

First 5 tracks:
  - Al James - Schoolboy Facination
  - AM Contra - Heart Peripheral
  - Angels In Amplifiers - I'm Alright
  - Arise - Run Run Run
  - Ben Carrigan - We'll Talk About It All Tonight

Last 5 tracks:
  

## Cell 5: Existing HT-Demucs Prediction Audit

This cell audits the existing EXP002 prediction directory against the verified 50-track MUSDB18 test set.

For each MUSDB18 test track, the notebook checks whether the four expected HT-Demucs separated sources are already present:

- `vocals.wav`
- `drums.wav`
- `bass.wav`
- `other.wav`

A track is considered complete only when all four prediction files are present.

The result determines whether the later inference stage should:

1. Skip inference entirely when all 50 tracks are complete.
2. Run inference only for missing/incomplete tracks.
3. Run inference for all tracks when no valid predictions are present.

Existing prediction files will not be overwritten merely because this audit is being performed.

In [5]:
# ============================================================
# CELL 5: EXISTING HT-DEMUCS PREDICTION AUDIT
# ============================================================

from pathlib import Path

print("=" * 70)
print("HT-DEMUCS PREDICTION AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# Expected HT-Demucs output sources
# ------------------------------------------------------------

EXPECTED_SOURCES = [
    "vocals.wav",
    "drums.wav",
    "bass.wav",
    "other.wav",
]

EXPECTED_SOURCE_SET = set(EXPECTED_SOURCES)

# ------------------------------------------------------------
# Verify prediction directory
# ------------------------------------------------------------

if not PREDICTIONS_DIR.exists():
    raise FileNotFoundError(
        f"Prediction directory does not exist:\n{PREDICTIONS_DIR}"
    )

print(f"\nPrediction directory:")
print(f"  {PREDICTIONS_DIR}")

print(f"\nExpected test tracks:")
print(f"  {len(MUSDB18_TRACK_NAMES)}")

print(f"Expected sources per track:")
print(f"  {len(EXPECTED_SOURCES)}")

print(f"Expected total prediction WAV files:")
print(f"  {len(MUSDB18_TRACK_NAMES) * len(EXPECTED_SOURCES)}")

# ------------------------------------------------------------
# Audit every MUSDB18 test track
# ------------------------------------------------------------

complete_tracks = []
incomplete_tracks = []
missing_track_directories = []

prediction_file_count = 0

audit_results = []

for track_dir in MUSDB18_TRACK_DIRS:

    track_name = track_dir.name
    prediction_track_dir = PREDICTIONS_DIR / track_name

    # --------------------------------------------------------
    # Prediction track directory missing
    # --------------------------------------------------------

    if not prediction_track_dir.exists():
        missing_track_directories.append(track_name)

        audit_results.append({
            "track": track_name,
            "prediction_directory": False,
            "vocals": False,
            "drums": False,
            "bass": False,
            "other": False,
            "complete": False,
        })

        continue

    # --------------------------------------------------------
    # Check expected source files
    # --------------------------------------------------------

    source_status = {}

    for source_file in EXPECTED_SOURCES:
        exists = (prediction_track_dir / source_file).is_file()
        source_name = Path(source_file).stem
        source_status[source_name] = exists

        if exists:
            prediction_file_count += 1

    is_complete = all(source_status.values())

    if is_complete:
        complete_tracks.append(track_name)
    else:
        incomplete_tracks.append(track_name)

    audit_results.append({
        "track": track_name,
        "prediction_directory": True,
        "vocals": source_status["vocals"],
        "drums": source_status["drums"],
        "bass": source_status["bass"],
        "other": source_status["other"],
        "complete": is_complete,
    })

# ------------------------------------------------------------
# Calculate totals
# ------------------------------------------------------------

total_tracks = len(MUSDB18_TRACK_NAMES)
complete_count = len(complete_tracks)
incomplete_count = len(incomplete_tracks)
missing_directory_count = len(missing_track_directories)

expected_prediction_files = total_tracks * len(EXPECTED_SOURCES)

# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PREDICTION AUDIT SUMMARY")
print("=" * 70)

print(f"\nTotal MUSDB18 test tracks:")
print(f"  {total_tracks}")

print(f"\nComplete prediction tracks:")
print(f"  {complete_count} / {total_tracks}")

print(f"\nIncomplete prediction tracks:")
print(f"  {incomplete_count} / {total_tracks}")

print(f"\nMissing prediction directories:")
print(f"  {missing_directory_count}")

print(f"\nPrediction WAV files found:")
print(f"  {prediction_file_count} / {expected_prediction_files}")

# ------------------------------------------------------------
# Detailed incomplete-track report
# ------------------------------------------------------------

if incomplete_tracks:

    print("\n" + "=" * 70)
    print("INCOMPLETE TRACKS")
    print("=" * 70)

    for track_name in incomplete_tracks:

        result = next(
            item for item in audit_results
            if item["track"] == track_name
        )

        missing_sources = [
            source
            for source in ["vocals", "drums", "bass", "other"]
            if not result[source]
        ]

        if not result["prediction_directory"]:
            print(f"\n✗ {track_name}")
            print("    Prediction directory missing")
        else:
            print(f"\n⚠ {track_name}")
            print(f"    Missing: {', '.join(missing_sources)}")

else:

    print("\n" + "=" * 70)
    print("ALL TRACKS COMPLETE")
    print("=" * 70)

    print(
        "\n✓ All 50 MUSDB18 test tracks have all four "
        "HT-Demucs prediction files."
    )

# ------------------------------------------------------------
# Determine inference requirement
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INFERENCE DECISION")
print("=" * 70)

if complete_count == total_tracks:
    INFERENCE_REQUIRED = False
    INFERENCE_MODE = "SKIP_ALL"

    print("\n✓ INFERENCE NOT REQUIRED")
    print("  All 50 tracks already have complete predictions.")
    print("  The later inference stage will be skipped.")

elif complete_count == 0:
    INFERENCE_REQUIRED = True
    INFERENCE_MODE = "RUN_ALL"

    print("\n⚠ INFERENCE REQUIRED FOR ALL TRACKS")
    print("  No complete prediction tracks were found.")

else:
    INFERENCE_REQUIRED = True
    INFERENCE_MODE = "RUN_MISSING"

    print("\n⚠ PARTIAL INFERENCE REQUIRED")
    print(f"  {incomplete_count} track(s) are incomplete.")
    print("  The later inference stage will process only those tracks.")

# ------------------------------------------------------------
# Final machine-readable state
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AUDIT STATE")
print("=" * 70)

print(f"\nINFERENCE_REQUIRED = {INFERENCE_REQUIRED}")
print(f"INFERENCE_MODE     = '{INFERENCE_MODE}'")
print(f"COMPLETE_TRACKS    = {complete_count}")
print(f"INCOMPLETE_TRACKS  = {incomplete_count}")
print(f"PREDICTION_FILES   = {prediction_file_count}")
print(f"EXPECTED_FILES     = {expected_prediction_files}")

print("\n" + "=" * 70)
print("✓ PREDICTION AUDIT COMPLETE")
print("=" * 70)

HT-DEMUCS PREDICTION AUDIT

Prediction directory:
  G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\predictions

Expected test tracks:
  50
Expected sources per track:
  4
Expected total prediction WAV files:
  200

PREDICTION AUDIT SUMMARY

Total MUSDB18 test tracks:
  50

Complete prediction tracks:
  50 / 50

Incomplete prediction tracks:
  0 / 50

Missing prediction directories:
  0

Prediction WAV files found:
  200 / 200

ALL TRACKS COMPLETE

✓ All 50 MUSDB18 test tracks have all four HT-Demucs prediction files.

INFERENCE DECISION

✓ INFERENCE NOT REQUIRED
  All 50 tracks already have complete predictions.
  The later inference stage will be skipped.

AUDIT STATE

INFERENCE_REQUIRED = False
INFERENCE_MODE     = 'SKIP_ALL'
COMPLETE_TRACKS    = 50
INCOMPLETE_TRACKS  = 0
PREDICTION_FILES   = 200
EXPECTED_FILES     = 200

✓ PREDICTION AUDIT COMPLETE


# Cell 6: HT-Demucs Environment Preparation

This cell verifies that the local `iks_eval` environment contains the required
Demucs package for HT-Demucs inference.

The current EXP002 prediction audit has already confirmed that all 50 MUSDB18
test tracks have complete HT-Demucs predictions, so no inference is required
for the current experiment.

However, Demucs must still be available in the local environment because:

- it is required for reproducing HT-Demucs inference;
- it may be required for future experiments such as EXP003;
- the notebook should remain self-contained and reproducible;
- future missing or incomplete predictions can be regenerated without rebuilding
  the environment.

If Demucs is already installed, its installed version is reported.

If Demucs is not installed, this cell installs the required current Demucs
release and then verifies the installation.

This cell does NOT run inference.

In [6]:
# ============================================================
# HT-DEMUCS ENVIRONMENT PREPARATION
# ============================================================

import sys
import subprocess
import importlib.util
import re

print("=" * 70)
print("HT-DEMUCS ENVIRONMENT PREPARATION")
print("=" * 70)

print("\nPython executable:")
print(f"  {sys.executable}")

print("\nPython version:")
print(f"  {sys.version.split()[0]}")

# ------------------------------------------------------------
# Expected Demucs version
# ------------------------------------------------------------

REQUIRED_DEMUCS_VERSION = "4.1.0"

print("\n" + "=" * 70)
print("DEMucs PACKAGE CHECK")
print("=" * 70)

demucs_spec = importlib.util.find_spec("demucs")

if demucs_spec is not None:
    print("\n✓ Demucs Python module is installed.")

    try:
        import demucs
        installed_version = getattr(demucs, "__version__", "unknown")
    except Exception:
        installed_version = "unknown"

    print(f"  Installed version: {installed_version}")

else:
    print("\n⚠ Demucs Python module is not installed.")
    installed_version = None


# ------------------------------------------------------------
# Install Demucs if missing
# ------------------------------------------------------------

if demucs_spec is None:

    print("\n" + "=" * 70)
    print("DEMucs INSTALLATION")
    print("=" * 70)

    print("\nInstalling Demucs...")
    print(f"Requested version: {REQUIRED_DEMUCS_VERSION}")

    install_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        f"demucs=={REQUIRED_DEMUCS_VERSION}"
    ]

    result = subprocess.run(
        install_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            "Demucs installation failed. "
            "Review the pip output above before continuing."
        )

    print("\n✓ Demucs installation command completed.")

    # Re-check after installation
    demucs_spec = importlib.util.find_spec("demucs")

    if demucs_spec is None:
        raise RuntimeError(
            "Demucs installation appeared to complete, "
            "but the demucs module cannot be imported."
        )

    import demucs
    installed_version = getattr(demucs, "__version__", "unknown")

    print(f"✓ Demucs module detected after installation.")
    print(f"  Installed version: {installed_version}")


# ------------------------------------------------------------
# Demucs command verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DEMucs COMMAND CHECK")
print("=" * 70)

command_result = subprocess.run(
    [sys.executable, "-m", "demucs", "--help"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

if command_result.returncode == 0:
    demucs_command_available = True
    print("\n✓ Demucs command is operational.")

else:
    demucs_command_available = False
    print("\n✗ Demucs command check failed.")
    print("\nOutput:")
    print(command_result.stdout)

    raise RuntimeError(
        "Demucs is installed but the Demucs command could not be executed."
    )


# ------------------------------------------------------------
# PyTorch / CUDA verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PYTORCH / CUDA CHECK")
print("=" * 70)

import torch

print(f"\nPyTorch:")
print(f"  {torch.__version__}")

print(f"\nCUDA available:")
print(f"  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print("\nGPU:")
    print(f"  {torch.cuda.get_device_name(0)}")

    print("\nTorch CUDA:")
    print(f"  {torch.version.cuda}")

else:
    print("\n⚠ CUDA GPU is not available.")


# ------------------------------------------------------------
# Final environment state
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HT-DEMUCS ENVIRONMENT RESULT")
print("=" * 70)

print(f"\nDemucs installed:")
print(f"  {demucs_spec is not None}")

print(f"\nDemucs version:")
print(f"  {installed_version}")

print(f"\nDemucs command available:")
print(f"  {demucs_command_available}")

print("\n" + "=" * 70)
print("✓ HT-DEMUCS ENVIRONMENT PREPARATION COMPLETE")
print("=" * 70)

print("\nNo inference was performed by this cell.")
print("Inference remains controlled by the prediction audit in Cell 5.")

HT-DEMUCS ENVIRONMENT PREPARATION

Python executable:
  C:\IKS_Local_Runtime\iks_eval\Scripts\python.exe

Python version:
  3.12.10

DEMucs PACKAGE CHECK

✓ Demucs Python module is installed.
  Installed version: 4.1.0

DEMucs COMMAND CHECK

✓ Demucs command is operational.

PYTORCH / CUDA CHECK

PyTorch:
  2.11.0+cu128

CUDA available:
  True

GPU:
  NVIDIA GeForce RTX 3050 Laptop GPU

Torch CUDA:
  12.8

HT-DEMUCS ENVIRONMENT RESULT

Demucs installed:
  True

Demucs version:
  4.1.0

Demucs command available:
  True

✓ HT-DEMUCS ENVIRONMENT PREPARATION COMPLETE

No inference was performed by this cell.
Inference remains controlled by the prediction audit in Cell 5.


# Cell 7: HT-Demucs Inference and Prediction Completion

This cell executes the HT-Demucs inference stage for the MUSDB18 test set.

The prediction audit performed in Cell 5 determines whether inference is
actually required.

Inference follows these rules:

1. If all 50 MUSDB18 test tracks already contain all four expected HT-Demucs
   prediction sources, inference is skipped completely.

2. If some tracks are incomplete, inference is performed only for those
   incomplete tracks.

3. If all tracks are missing predictions, inference is performed for all
   50 test tracks.

4. Existing complete prediction tracks are never unnecessarily regenerated.

The four expected HT-Demucs sources for each track are:

- drums.wav
- bass.wav
- other.wav
- vocals.wav

The inference output is stored directly inside the existing EXP002
`predictions` directory, using one directory per MUSDB18 test track.

This cell uses the locally verified Demucs 4.1.0 installation and the local
RTX 3050 GPU when CUDA is available.

No evaluation metrics are calculated in this cell.

After inference, a final prediction audit verifies that every required track
contains all four expected source files before the notebook proceeds to the
evaluation stage.

In [7]:
# ============================================================
# HT-DEMUCS INFERENCE AND PREDICTION COMPLETION
# ============================================================

from pathlib import Path
import subprocess
import sys
import torch
import time

print("=" * 70)
print("HT-DEMUCS INFERENCE STAGE")
print("=" * 70)

# ------------------------------------------------------------
# Verify required state from previous cells
# ------------------------------------------------------------

required_variables = [
    "MUSDB18_TEST_DIR",
    "PREDICTIONS_DIR",
    "INFERENCE_REQUIRED",
    "INFERENCE_MODE",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Required variables from previous cells are missing:\n"
        + "\n".join(f"  - {name}" for name in missing_variables)
    )

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SOURCE_NAMES = [
    "drums",
    "bass",
    "other",
    "vocals",
]

EXPECTED_SOURCE_FILES = {
    source: f"{source}.wav"
    for source in SOURCE_NAMES
}

TEST_DIR = Path(MUSDB18_TEST_DIR)
OUTPUT_DIR = Path(PREDICTIONS_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Runtime information
# ------------------------------------------------------------

print("\nRuntime:")
print(f"  Python: {sys.executable}")

print("\nPyTorch:")
print(f"  Version: {torch.__version__}")

print("\nCUDA:")
print(f"  Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    print("  GPU: Not available")
    print("  Inference will use CPU.")

print("\nDemucs:")
print("  Version: 4.1.0")

# ------------------------------------------------------------
# Discover test tracks
# ------------------------------------------------------------

test_tracks = sorted(
    [
        path for path in TEST_DIR.iterdir()
        if path.is_dir()
    ],
    key=lambda p: p.name.lower()
)

print("\n" + "=" * 70)
print("TEST-SET INFERENCE PLAN")
print("=" * 70)

print(f"\nMUSDB18 test tracks discovered: {len(test_tracks)}")
print(f"Expected tracks: 50")

if len(test_tracks) != 50:
    raise RuntimeError(
        f"Expected 50 MUSDB18 test tracks, but found {len(test_tracks)}."
    )

# ------------------------------------------------------------
# Determine incomplete tracks directly from filesystem
# ------------------------------------------------------------

incomplete_tracks = []

for track_dir in test_tracks:

    prediction_dir = OUTPUT_DIR / track_dir.name

    missing_sources = []

    for source_name, filename in EXPECTED_SOURCE_FILES.items():

        source_path = prediction_dir / filename

        if not source_path.is_file():
            missing_sources.append(filename)

    if missing_sources:
        incomplete_tracks.append(
            {
                "track": track_dir,
                "prediction_dir": prediction_dir,
                "missing_sources": missing_sources,
            }
        )

# ------------------------------------------------------------
# Print current prediction state
# ------------------------------------------------------------

complete_count = len(test_tracks) - len(incomplete_tracks)

print(f"\nComplete prediction tracks:")
print(f"  {complete_count} / {len(test_tracks)}")

print(f"\nIncomplete prediction tracks:")
print(f"  {len(incomplete_tracks)} / {len(test_tracks)}")

# ------------------------------------------------------------
# Safety check against Cell 5 decision
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INFERENCE DECISION")
print("=" * 70)

if len(incomplete_tracks) == 0:

    print("\n✓ ALL PREDICTIONS ARE COMPLETE")
    print("  No inference is required.")
    print("  Existing HT-Demucs predictions will not be modified.")

    INFERENCE_EXECUTED = False
    INFERENCE_TRACKS = []

else:

    INFERENCE_EXECUTED = True
    INFERENCE_TRACKS = [
        item["track"].name
        for item in incomplete_tracks
    ]

    print("\n⚠ INFERENCE IS REQUIRED")
    print(f"  Tracks requiring inference: {len(incomplete_tracks)}")

    if len(incomplete_tracks) == len(test_tracks):
        print("  Mode: FULL INFERENCE")
    else:
        print("  Mode: PARTIAL INFERENCE")

    print("\nTracks requiring inference:")

    for index, item in enumerate(incomplete_tracks, start=1):
        print(
            f"  [{index:02d}/{len(incomplete_tracks):02d}] "
            f"{item['track'].name}"
        )

# ------------------------------------------------------------
# Execute inference only when required
# ------------------------------------------------------------

if INFERENCE_EXECUTED:

    print("\n" + "=" * 70)
    print("STARTING HT-DEMUCS INFERENCE")
    print("=" * 70)

    inference_start = time.time()

    for index, item in enumerate(incomplete_tracks, start=1):

        track_dir = item["track"]
        prediction_dir = item["prediction_dir"]

        prediction_dir.mkdir(parents=True, exist_ok=True)

        print("\n" + "-" * 70)
        print(
            f"TRACK {index}/{len(incomplete_tracks)}"
        )
        print("-" * 70)

        print(f"Track:")
        print(f"  {track_dir.name}")

        print(f"\nInput:")
        print(f"  {track_dir}")

        print(f"\nOutput:")
        print(f"  {prediction_dir}")

        print(f"\nMissing files:")
        for filename in item["missing_sources"]:
            print(f"  - {filename}")

        # ----------------------------------------------------
        # Locate mixture.wav
        # ----------------------------------------------------

        mixture_path = track_dir / "mixture.wav"

        if not mixture_path.is_file():

            # Some MUSDB18 layouts may use the track directory
            # itself as the input source. We do not silently guess.
            raise FileNotFoundError(
                f"Required mixture.wav not found for track:\n"
                f"{track_dir}"
            )

        # ----------------------------------------------------
        # Demucs command
        # ----------------------------------------------------
        #
        # `htdemucs` is the standard Hybrid Transformer Demucs
        # model identifier used by Demucs.
        #
        # We use the local Python executable so that the command
        # definitely runs inside the verified iks_eval environment.
        #
        # --out specifies the EXP002 prediction directory.
        # --name htdemucs selects the HT-Demucs model.
        # --device selects CUDA when available.
        # ----------------------------------------------------

        command = [
            sys.executable,
            "-m",
            "demucs",
            "--name",
            "htdemucs",
            "--out",
            str(OUTPUT_DIR),
            "--device",
            DEVICE,
            str(mixture_path),
        ]

        print("\nExecuting:")
        print("  " + " ".join(f'"{arg}"' if " " in arg else arg
                              for arg in command))

        track_start = time.time()

        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

        elapsed = time.time() - track_start

        print("\nDemucs output:")
        print(result.stdout)

        if result.returncode != 0:

            raise RuntimeError(
                f"HT-Demucs inference failed for:\n"
                f"{track_dir.name}\n\n"
                f"Demucs output:\n"
                f"{result.stdout}"
            )

        print(
            f"\n✓ Inference completed in {elapsed / 60:.2f} minutes."
        )

    total_elapsed = time.time() - inference_start

    print("\n" + "=" * 70)
    print("HT-DEMUCS INFERENCE FINISHED")
    print("=" * 70)

    print(
        f"\nTotal inference time: "
        f"{total_elapsed / 60:.2f} minutes"
    )

else:

    print("\n" + "=" * 70)
    print("HT-DEMUCS INFERENCE SKIPPED")
    print("=" * 70)

    print("\n✓ Existing predictions will be preserved.")
    print("✓ No Demucs inference was executed.")

# ------------------------------------------------------------
# POST-INFERENCE VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("POST-INFERENCE PREDICTION VALIDATION")
print("=" * 70)

validation_failures = []
total_prediction_files = 0

for track_dir in test_tracks:

    prediction_dir = OUTPUT_DIR / track_dir.name

    missing_sources = []

    for source_name, filename in EXPECTED_SOURCE_FILES.items():

        source_path = prediction_dir / filename

        if source_path.is_file():
            total_prediction_files += 1
        else:
            missing_sources.append(filename)

    if missing_sources:

        validation_failures.append(
            {
                "track": track_dir.name,
                "missing": missing_sources,
            }
        )

expected_total_files = len(test_tracks) * len(SOURCE_NAMES)

print(f"\nTracks:")
print(f"  {len(test_tracks)}")

print(f"\nExpected prediction WAV files:")
print(f"  {expected_total_files}")

print(f"\nPrediction WAV files found:")
print(f"  {total_prediction_files}")

print(f"\nTracks with complete predictions:")
print(
    f"  {len(test_tracks) - len(validation_failures)} "
    f"/ {len(test_tracks)}"
)

print(f"\nTracks with missing predictions:")
print(
    f"  {len(validation_failures)} "
    f"/ {len(test_tracks)}"
)

# ------------------------------------------------------------
# Final validation result
# ------------------------------------------------------------

if validation_failures:

    print("\n" + "=" * 70)
    print("✗ POST-INFERENCE VALIDATION FAILED")
    print("=" * 70)

    print("\nIncomplete tracks:")

    for item in validation_failures:

        print(f"\n  {item['track']}")

        for filename in item["missing"]:
            print(f"    Missing: {filename}")

    raise RuntimeError(
        "Prediction validation failed. "
        "Do not proceed to evaluation until all required "
        "prediction files are present."
    )

else:

    print("\n" + "=" * 70)
    print("✓ POST-INFERENCE VALIDATION PASSED")
    print("=" * 70)

    print("\n✓ All 50 MUSDB18 test tracks are complete.")
    print("✓ All 200 expected HT-Demucs prediction WAV files exist.")
    print("✓ Prediction directory is ready for evaluation.")

HT-DEMUCS INFERENCE STAGE

Runtime:
  Python: C:\IKS_Local_Runtime\iks_eval\Scripts\python.exe

PyTorch:
  Version: 2.11.0+cu128

CUDA:
  Available: True
  GPU: NVIDIA GeForce RTX 3050 Laptop GPU

Demucs:
  Version: 4.1.0

TEST-SET INFERENCE PLAN

MUSDB18 test tracks discovered: 50
Expected tracks: 50

Complete prediction tracks:
  50 / 50

Incomplete prediction tracks:
  0 / 50

INFERENCE DECISION

✓ ALL PREDICTIONS ARE COMPLETE
  No inference is required.
  Existing HT-Demucs predictions will not be modified.

HT-DEMUCS INFERENCE SKIPPED

✓ Existing predictions will be preserved.
✓ No Demucs inference was executed.

POST-INFERENCE PREDICTION VALIDATION

Tracks:
  50

Expected prediction WAV files:
  200

Prediction WAV files found:
  200

Tracks with complete predictions:
  50 / 50

Tracks with missing predictions:
  0 / 50

✓ POST-INFERENCE VALIDATION PASSED

✓ All 50 MUSDB18 test tracks are complete.
✓ All 200 expected HT-Demucs prediction WAV files exist.
✓ Prediction directory is

In [8]:
# ============================================================
# CELL 8: GROUND-TRUTH / PREDICTION ALIGNMENT AUDIT
# ============================================================
#
# Purpose:
#   Verify that the MUSDB18 reference sources and the existing
#   HT-Demucs predictions are correctly aligned before objective
#   evaluation.
#
# This cell does NOT calculate metrics and does NOT modify audio.
# It only validates the evaluation inputs.
# ============================================================

from pathlib import Path

print("=" * 70)
print("GROUND-TRUTH / PREDICTION ALIGNMENT AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# Expected MUSDB18 source structure
# ------------------------------------------------------------

GROUND_TRUTH_SOURCES = {
    "drums.wav",
    "bass.wav",
    "vocals.wav",
    "other.wav",
    "mixture.wav",
}

PREDICTION_SOURCES = {
    "drums.wav",
    "bass.wav",
    "vocals.wav",
    "other.wav",
}

# ------------------------------------------------------------
# Discover directories
# ------------------------------------------------------------

ground_truth_tracks = {
    p.name: p
    for p in MUSDB18_TEST_DIR.iterdir()
    if p.is_dir()
}

prediction_tracks = {
    p.name: p
    for p in PREDICTIONS_DIR.iterdir()
    if p.is_dir()
}

print("\nGround-truth tracks:", len(ground_truth_tracks))
print("Prediction tracks:", len(prediction_tracks))

# ------------------------------------------------------------
# Track-name alignment
# ------------------------------------------------------------

ground_truth_names = set(ground_truth_tracks.keys())
prediction_names = set(prediction_tracks.keys())

missing_predictions = sorted(
    ground_truth_names - prediction_names
)

unexpected_predictions = sorted(
    prediction_names - ground_truth_names
)

print("\n" + "=" * 70)
print("TRACK-NAME ALIGNMENT")
print("=" * 70)

if not missing_predictions:
    print("✓ No MUSDB18 tracks are missing from predictions.")
else:
    print(f"✗ Missing prediction directories: {len(missing_predictions)}")
    for name in missing_predictions:
        print("  -", name)

if not unexpected_predictions:
    print("✓ No unexpected prediction directories detected.")
else:
    print(f"⚠ Unexpected prediction directories: {len(unexpected_predictions)}")
    for name in unexpected_predictions:
        print("  -", name)

# ------------------------------------------------------------
# Source-file alignment
# ------------------------------------------------------------

ground_truth_source_errors = []
prediction_source_errors = []

for track_name in sorted(ground_truth_names & prediction_names):

    gt_dir = ground_truth_tracks[track_name]
    pred_dir = prediction_tracks[track_name]

    gt_files = {
        p.name.lower()
        for p in gt_dir.glob("*.wav")
    }

    pred_files = {
        p.name.lower()
        for p in pred_dir.glob("*.wav")
    }

    missing_gt = sorted(
        {x.lower() for x in GROUND_TRUTH_SOURCES} - gt_files
    )

    missing_pred = sorted(
        {x.lower() for x in PREDICTION_SOURCES} - pred_files
    )

    if missing_gt:
        ground_truth_source_errors.append(
            (track_name, missing_gt)
        )

    if missing_pred:
        prediction_source_errors.append(
            (track_name, missing_pred)
        )

# ------------------------------------------------------------
# Report source validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GROUND-TRUTH SOURCE VALIDATION")
print("=" * 70)

if not ground_truth_source_errors:
    print("✓ All common test tracks contain:")
    print("  - mixture.wav")
    print("  - vocals.wav")
    print("  - drums.wav")
    print("  - bass.wav")
    print("  - other.wav")
else:
    print(
        f"✗ Ground-truth source errors: "
        f"{len(ground_truth_source_errors)} tracks"
    )

    for track_name, missing in ground_truth_source_errors:
        print(f"\n  {track_name}")
        print("    Missing:", ", ".join(missing))

print("\n" + "=" * 70)
print("PREDICTION SOURCE VALIDATION")
print("=" * 70)

if not prediction_source_errors:
    print("✓ All common test tracks contain the four HT-Demucs sources:")
    print("  - vocals.wav")
    print("  - drums.wav")
    print("  - bass.wav")
    print("  - other.wav")
else:
    print(
        f"✗ Prediction source errors: "
        f"{len(prediction_source_errors)} tracks"
    )

    for track_name, missing in prediction_source_errors:
        print(f"\n  {track_name}")
        print("    Missing:", ", ".join(missing))

# ------------------------------------------------------------
# Final alignment result
# ------------------------------------------------------------

ALIGNMENT_VALID = (
    len(missing_predictions) == 0
    and len(ground_truth_source_errors) == 0
    and len(prediction_source_errors) == 0
)

print("\n" + "=" * 70)
print("ALIGNMENT AUDIT RESULT")
print("=" * 70)

if ALIGNMENT_VALID:
    print("✓ GROUND-TRUTH / PREDICTION ALIGNMENT PASSED")
    print()
    print("All 50 MUSDB18 test tracks are aligned.")
    print("All required reference sources are present.")
    print("All four HT-Demucs prediction sources are present.")
    print()
    print("The evaluation stage may proceed.")
else:
    print("✗ GROUND-TRUTH / PREDICTION ALIGNMENT FAILED")
    print()
    print("Evaluation must NOT proceed until the input mismatch is resolved.")

print("\n" + "=" * 70)

GROUND-TRUTH / PREDICTION ALIGNMENT AUDIT

Ground-truth tracks: 50
Prediction tracks: 50

TRACK-NAME ALIGNMENT
✓ No MUSDB18 tracks are missing from predictions.
✓ No unexpected prediction directories detected.

GROUND-TRUTH SOURCE VALIDATION
✓ All common test tracks contain:
  - mixture.wav
  - vocals.wav
  - drums.wav
  - bass.wav
  - other.wav

PREDICTION SOURCE VALIDATION
✓ All common test tracks contain the four HT-Demucs sources:
  - vocals.wav
  - drums.wav
  - bass.wav
  - other.wav

ALIGNMENT AUDIT RESULT
✓ GROUND-TRUTH / PREDICTION ALIGNMENT PASSED

All 50 MUSDB18 test tracks are aligned.
All required reference sources are present.
All four HT-Demucs prediction sources are present.

The evaluation stage may proceed.



# Cell 9 — Objective Evaluation: BSSEval v4 + SI-SDR

This cell performs the actual objective evaluation of the existing HT-Demucs
predictions against the MUSDB18 test-set references.

## Evaluation metrics

Two complementary evaluation families are used:

### 1. BSSEval v4

The `museval` implementation of BSSEval v4 is explicitly invoked to calculate:

- SDR — Signal-to-Distortion Ratio
- SIR — Source-to-Interference Ratio
- SAR — Sources-to-Artifacts Ratio
- ISR — Image-to-Spatial Ratio

BSSEval v4 is used with time-invariant distortion filters and without
per-frame filter re-estimation. This avoids the older BSSEval source-wrapper
configuration and provides the standardized source-separation evaluation
used by the MUSDB/SigSep ecosystem.

### 2. SI-SDR

Scale-Invariant Signal-to-Distortion Ratio is calculated separately using
the standard scale-invariant projection formulation.

SI-SDR is intentionally not substituted for BSSEval SDR. The two metrics
measure separation quality differently and are therefore retained as
separate reported metrics.

## Evaluation design

The evaluation is performed sequentially:

    MUSDB18 track
        ↓
    Load reference sources
        ↓
    Load HT-Demucs predictions
        ↓
    Validate sample rate / channels / lengths
        ↓
    Align reference and estimate arrays
        ↓
    BSSEval v4
        ├── SDR
        ├── SIR
        ├── SAR
        └── ISR
        ↓
    SI-SDR
        ↓
    Store per-source results
        ↓
    Release track arrays
        ↓
    Process next track

Only one MUSDB18 track is evaluated at a time. The complete 50-track dataset
is therefore never loaded into RAM simultaneously.

## Source order

The evaluation uses the fixed MUSDB18 source order:

1. vocals
2. drums
3. bass
4. other

No source permutation is necessary because the HT-Demucs prediction filenames
are already aligned with the corresponding MUSDB18 reference source names.

## Output policy

This cell writes evaluation results into the directories already validated
by Cell 3:

- `metrics/`  → machine-readable metric outputs
- `logs/`     → evaluation progress and errors
- `reports/`  → reserved for later aggregate reports
- `plots/`    → reserved for later visualization cells

The existing HT-Demucs predictions are read-only inputs and are never modified.

The evaluation is CPU-based. CUDA is not required for BSSEval or SI-SDR and
no GPU tensor containing the complete test set is created.

The cell is designed to be restart-safe. Results are written incrementally
so that a later failure does not require previously completed tracks to be
recalculated.

In [9]:
# ================================================================
# CELL 9 — OBJECTIVE EVALUATION
# BSSEval v4 + SI-SDR
# ================================================================

import os
import sys
import gc
import json
import math
import time
import traceback
import warnings
import importlib.util
from pathlib import Path
from datetime import datetime

import numpy as np
import soundfile as sf


# ----------------------------------------------------------------
# 1. DEPENDENCY CHECK
# ----------------------------------------------------------------

print("=" * 70)
print("OBJECTIVE EVALUATION — BSSEVAL v4 + SI-SDR")
print("=" * 70)

print("\nPython:")
print(sys.executable)

# museval is intentionally imported here rather than replacing
# BSSEval with a custom SDR/SIR/SAR implementation.
if importlib.util.find_spec("museval") is None:
    raise ImportError(
        "The 'museval' package is not installed in the local iks_eval "
        "environment. Install it before running Cell 9."
    )

import museval
from museval.metrics import bss_eval


print("\nDependency status:")
print(f"  museval: {getattr(museval, '__version__', 'installed')}")
print("  soundfile: installed")
print("  numpy: installed")

# ----------------------------------------------------------------
# 2. VERIFY VARIABLES CREATED BY PREVIOUS CELLS
# ----------------------------------------------------------------

required_variables = [
    "MUSDB18_TEST_DIR",
    "PREDICTIONS_DIR",
    "METRICS_DIR",
    "PLOTS_DIR",
    "REPORTS_DIR",
    "LOGS_DIR",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Required configuration variables are missing from previous cells:\n"
        + "\n".join(f"  - {name}" for name in missing_variables)
    )


MUSDB18_TEST_DIR = Path(MUSDB18_TEST_DIR)
PREDICTIONS_DIR = Path(PREDICTIONS_DIR)
METRICS_DIR = Path(METRICS_DIR)
PLOTS_DIR = Path(PLOTS_DIR)
REPORTS_DIR = Path(REPORTS_DIR)
LOGS_DIR = Path(LOGS_DIR)


# ----------------------------------------------------------------
# 3. EVALUATION CONFIGURATION
# ----------------------------------------------------------------

SOURCE_ORDER = [
    "vocals",
    "drums",
    "bass",
    "other",
]

EXPECTED_TRACK_COUNT = 50

# BSSEval v4 configuration.
#
# window=np.inf and hop=np.inf cause museval to evaluate the complete
# track rather than creating many overlapping evaluation windows.
#
# framewise_filters=False is the important choice here. It uses
# time-invariant distortion filters, avoiding the older framewise
# filter behavior and greatly reducing computational overhead.
BSSEVAL_WINDOW = np.inf
BSSEVAL_HOP = np.inf
BSSEVAL_FILTERS_LEN = 512
BSSEVAL_FRAMEWISE_FILTERS = False
BSSEVAL_COMPUTE_PERMUTATION = False
BSSEVAL_SOURCES_VERSION = False


# ----------------------------------------------------------------
# 4. OUTPUT FILES
# ----------------------------------------------------------------

METRICS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

TRACK_RESULTS_JSONL = METRICS_DIR / "per_track_metrics.jsonl"
TRACK_RESULTS_CSV = METRICS_DIR / "per_track_metrics.csv"
SUMMARY_JSON = METRICS_DIR / "evaluation_summary.json"

LOG_FILE = LOGS_DIR / "objective_evaluation.log"


# ----------------------------------------------------------------
# 5. LOGGING
# ----------------------------------------------------------------

def write_log(message):
    """Write a timestamped message to the evaluation log."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"

    print(line)

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")


write_log("Cell 9 objective evaluation started.")


# ----------------------------------------------------------------
# 6. AUDIO LOADING
# ----------------------------------------------------------------

def load_audio(path):
    """
    Load one WAV file as float32 NumPy audio.

    Returned shape:
        mono   -> (samples, 1)
        stereo -> (samples, channels)

    Audio remains on CPU. No full-test-set tensor is created.
    """

    audio, sample_rate = sf.read(
        str(path),
        dtype="float32",
        always_2d=True,
    )

    audio = np.asarray(audio, dtype=np.float32)

    return audio, int(sample_rate)


# ----------------------------------------------------------------
# 7. AUDIO VALIDATION / ALIGNMENT
# ----------------------------------------------------------------

def validate_and_align(reference, estimate, reference_sr, estimate_sr, source_name):
    """
    Validate sample rate, channels and duration.

    Small sample-count differences caused by file/container handling are
    handled by trimming both arrays to the common length.

    A large mismatch is treated as an evaluation error rather than being
    silently corrected.
    """

    if reference_sr != estimate_sr:
        raise ValueError(
            f"{source_name}: sample-rate mismatch: "
            f"reference={reference_sr}, estimate={estimate_sr}"
        )

    if reference.ndim != 2 or estimate.ndim != 2:
        raise ValueError(
            f"{source_name}: unexpected audio dimensions: "
            f"reference={reference.shape}, estimate={estimate.shape}"
        )

    if reference.shape[1] != estimate.shape[1]:
        raise ValueError(
            f"{source_name}: channel mismatch: "
            f"reference={reference.shape[1]}, "
            f"estimate={estimate.shape[1]}"
        )

    ref_len = reference.shape[0]
    est_len = estimate.shape[0]

    if ref_len == 0 or est_len == 0:
        raise ValueError(
            f"{source_name}: empty reference or estimate."
        )

    difference = abs(ref_len - est_len)

    # Small boundary differences can safely be aligned by trimming.
    # Large differences indicate an actual evaluation problem.
    max_allowed_difference = max(
        4096,
        int(0.01 * max(ref_len, est_len))
    )

    if difference > max_allowed_difference:
        raise ValueError(
            f"{source_name}: excessive length mismatch: "
            f"reference={ref_len}, estimate={est_len}, "
            f"difference={difference} samples"
        )

    common_length = min(ref_len, est_len)

    reference = reference[:common_length]
    estimate = estimate[:common_length]

    return reference, estimate


# ----------------------------------------------------------------
# 8. SI-SDR IMPLEMENTATION
# ----------------------------------------------------------------

def calculate_si_sdr(reference, estimate, zero_mean=False, eps=1e-10):
    """
    Calculate Scale-Invariant Signal-to-Distortion Ratio.

    Standard projection:

        alpha = <s, x> / <s, s>

        target = alpha * s

        noise = x - target

        SI-SDR = 10 log10(
            ||target||^2 / ||noise||^2
        )

    Input:
        reference : (samples, channels)
        estimate  : (samples, channels)

    The score is calculated independently for each channel and then
    averaged across channels.

    No Torch tensor is created, keeping this stage CPU/RAM efficient.
    """

    reference = np.asarray(reference, dtype=np.float64)
    estimate = np.asarray(estimate, dtype=np.float64)

    if zero_mean:
        reference = reference - np.mean(reference, axis=0, keepdims=True)
        estimate = estimate - np.mean(estimate, axis=0, keepdims=True)

    scores = []

    for channel in range(reference.shape[1]):

        s = reference[:, channel]
        x = estimate[:, channel]

        s_energy = np.dot(s, s)

        if s_energy <= eps:
            scores.append(np.nan)
            continue

        alpha = np.dot(x, s) / (s_energy + eps)

        target = alpha * s
        residual = x - target

        target_energy = np.dot(target, target)
        residual_energy = np.dot(residual, residual)

        if residual_energy <= eps:
            scores.append(np.inf)
        else:
            score = 10.0 * np.log10(
                max(target_energy, eps)
                / max(residual_energy, eps)
            )
            scores.append(float(score))

    if not scores:
        return np.nan

    return float(np.nanmean(scores))


# ----------------------------------------------------------------
# 9. EXISTING RESULT DISCOVERY
# ----------------------------------------------------------------

completed_track_results = set()

if TRACK_RESULTS_JSONL.exists():
    try:
        with open(TRACK_RESULTS_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                try:
                    record = json.loads(line)

                    if record.get("status") == "success":
                        track_name = record.get("track")

                        if track_name:
                            completed_track_results.add(track_name)

                except json.JSONDecodeError:
                    continue

    except Exception as exc:
        write_log(
            f"Warning: could not read existing metric JSONL: {exc}"
        )


# ----------------------------------------------------------------
# 10. DISCOVER TEST TRACKS
# ----------------------------------------------------------------

test_tracks = sorted(
    [
        path
        for path in MUSDB18_TEST_DIR.iterdir()
        if path.is_dir()
    ],
    key=lambda p: p.name.lower()
)


if len(test_tracks) != EXPECTED_TRACK_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_TRACK_COUNT} MUSDB18 test tracks, "
        f"but discovered {len(test_tracks)}."
    )


print("\n" + "=" * 70)
print("EVALUATION PLAN")
print("=" * 70)

print(f"\nMUSDB18 test tracks: {len(test_tracks)}")
print(f"Sources per track: {len(SOURCE_ORDER)}")
print("BSSEval: BSSEval v4")
print("BSSEval framewise filters: False")
print("BSSEval permutation: False")
print("SI-SDR: separate calculation")
print("Evaluation mode: one track at a time")


# ----------------------------------------------------------------
# 11. TRACK EVALUATION
# ----------------------------------------------------------------

all_results = []
failed_results = []

evaluation_start = time.time()


for track_index, track_dir in enumerate(test_tracks, start=1):

    track_name = track_dir.name

    print("\n" + "-" * 70)
    print(
        f"TRACK {track_index}/{len(test_tracks)}: "
        f"{track_name}"
    )
    print("-" * 70)

    # ------------------------------------------------------------
    # Restart-safe behavior
    #
    # If a valid successful result already exists in JSONL, skip it.
    # This is useful if Cell 9 is interrupted and restarted.
    # ------------------------------------------------------------

    if track_name in completed_track_results:
        print("✓ Existing successful evaluation found.")
        print("  Skipping recalculation for this track.")
        continue


    track_start = time.time()

    reference_sources = []
    estimated_sources = []

    sample_rate = None

    try:

        # --------------------------------------------------------
        # Load exactly one track
        # --------------------------------------------------------

        for source_name in SOURCE_ORDER:

            reference_path = track_dir / f"{source_name}.wav"
            estimate_path = (
                PREDICTIONS_DIR
                / track_name
                / f"{source_name}.wav"
            )

            if not reference_path.exists():
                raise FileNotFoundError(
                    f"Missing reference file: {reference_path}"
                )

            if not estimate_path.exists():
                raise FileNotFoundError(
                    f"Missing prediction file: {estimate_path}"
                )


            # Load reference.
            reference, reference_sr = load_audio(reference_path)

            # Load estimate.
            estimate, estimate_sr = load_audio(estimate_path)


            # Validate and align.
            reference, estimate = validate_and_align(
                reference,
                estimate,
                reference_sr,
                estimate_sr,
                source_name,
            )


            if sample_rate is None:
                sample_rate = reference_sr


            reference_sources.append(reference)
            estimated_sources.append(estimate)


            # Delete temporary variables immediately.
            del reference
            del estimate


        # --------------------------------------------------------
        # Convert source lists to BSSEval format
        #
        # museval expects:
        #
        #     (nsrc, nsamples, nchannels)
        #
        # This is only ONE track, not the entire test set.
        # --------------------------------------------------------

        reference_array = np.stack(
            reference_sources,
            axis=0,
        ).astype(np.float64, copy=False)

        estimated_array = np.stack(
            estimated_sources,
            axis=0,
        ).astype(np.float64, copy=False)


        # Free the source lists after stacking.
        del reference_sources
        del estimated_sources


        # --------------------------------------------------------
        # Final shape validation
        # --------------------------------------------------------

        if reference_array.shape != estimated_array.shape:
            raise ValueError(
                "Reference/prediction shape mismatch after stacking: "
                f"{reference_array.shape} vs "
                f"{estimated_array.shape}"
            )


        # --------------------------------------------------------
        # BSSEVAL v4
        # --------------------------------------------------------
        #
        # IMPORTANT:
        #
        # We explicitly invoke museval.metrics.bss_eval().
        #
        # This is the actual BSSEval v4 implementation.
        #
        # Returned order:
        #
        #     SDR, ISR, SIR, SAR, permutation
        #
        # Because source order is already known and aligned, permutation
        # calculation is disabled.
        #
        # framewise_filters=False means time-invariant filters.
        # --------------------------------------------------------

        write_log(
            f"{track_name}: running BSSEval v4."
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            sdr, isr, sir, sar, permutation = bss_eval(
                reference_array,
                estimated_array,
                window=BSSEVAL_WINDOW,
                hop=BSSEVAL_HOP,
                compute_permutation=BSSEVAL_COMPUTE_PERMUTATION,
                filters_len=BSSEVAL_FILTERS_LEN,
                framewise_filters=BSSEVAL_FRAMEWISE_FILTERS,
                bsseval_sources_version=BSSEVAL_SOURCES_VERSION,
            )


        # --------------------------------------------------------
        # SI-SDR
        # --------------------------------------------------------
        #
        # Calculate independently for each source.
        # --------------------------------------------------------

        write_log(
            f"{track_name}: calculating SI-SDR."
        )

        si_sdr_values = {}

        for source_index, source_name in enumerate(SOURCE_ORDER):

            si_sdr_values[source_name] = calculate_si_sdr(
                reference_array[source_index],
                estimated_array[source_index],
            )


        # --------------------------------------------------------
        # Extract BSSEval results
        # --------------------------------------------------------

        source_metrics = {}

        for source_index, source_name in enumerate(SOURCE_ORDER):

            source_metrics[source_name] = {
                "SDR_dB": float(np.nanmean(sdr[source_index])),
                "ISR_dB": float(np.nanmean(isr[source_index])),
                "SIR_dB": float(np.nanmean(sir[source_index])),
                "SAR_dB": float(np.nanmean(sar[source_index])),
                "SI_SDR_dB": si_sdr_values[source_name],
            }


        # --------------------------------------------------------
        # Track-level averages
        # --------------------------------------------------------

        track_metric_names = [
            "SDR_dB",
            "ISR_dB",
            "SIR_dB",
            "SAR_dB",
            "SI_SDR_dB",
        ]

        track_average = {}

        for metric_name in track_metric_names:

            values = [
                source_metrics[source][metric_name]
                for source in SOURCE_ORDER
            ]

            finite_values = [
                value
                for value in values
                if np.isfinite(value)
            ]

            if finite_values:
                track_average[metric_name] = float(
                    np.mean(finite_values)
                )
            else:
                track_average[metric_name] = None


        # --------------------------------------------------------
        # Create result record
        # --------------------------------------------------------

        elapsed = time.time() - track_start

        result = {
            "status": "success",
            "track": track_name,
            "track_index": track_index,
            "sample_rate": sample_rate,
            "sources": source_metrics,
            "track_average": track_average,
            "evaluation": {
                "bsseval": "BSSEval v4",
                "bsseval_window": "whole_track",
                "bsseval_framewise_filters": False,
                "bsseval_compute_permutation": False,
                "bsseval_filters_len": BSSEVAL_FILTERS_LEN,
                "si_sdr": "scale_invariant",
            },
            "elapsed_seconds": round(elapsed, 3),
            "timestamp": datetime.now().isoformat(),
        }


        # --------------------------------------------------------
        # Incremental write
        # --------------------------------------------------------

        with open(
            TRACK_RESULTS_JSONL,
            "a",
            encoding="utf-8",
        ) as f:

            f.write(
                json.dumps(
                    result,
                    allow_nan=True,
                )
                + "\n"
            )


        all_results.append(result)

        completed_track_results.add(track_name)

        print(f"✓ Track evaluation complete in {elapsed:.2f}s")

        for source_name in SOURCE_ORDER:

            metrics = source_metrics[source_name]

            print(
                f"  {source_name:>6}: "
                f"SDR={metrics['SDR_dB']:.3f} dB | "
                f"SIR={metrics['SIR_dB']:.3f} dB | "
                f"SAR={metrics['SAR_dB']:.3f} dB | "
                f"SI-SDR={metrics['SI_SDR_dB']:.3f} dB"
            )


        # --------------------------------------------------------
        # CRITICAL RAM CLEANUP
        # --------------------------------------------------------

        del reference_array
        del estimated_array
        del sdr
        del isr
        del sir
        del sar
        del permutation
        del si_sdr_values
        del source_metrics
        del track_average
        del result

        gc.collect()


    except Exception as exc:

        elapsed = time.time() - track_start

        error_record = {
            "status": "failed",
            "track": track_name,
            "track_index": track_index,
            "error": str(exc),
            "elapsed_seconds": round(elapsed, 3),
            "timestamp": datetime.now().isoformat(),
        }

        failed_results.append(error_record)

        with open(
            LOG_FILE,
            "a",
            encoding="utf-8",
        ) as f:

            f.write(
                "\n"
                + traceback.format_exc()
                + "\n"
            )

        write_log(
            f"ERROR: {track_name} failed: {exc}"
        )

        # Ensure the next track starts with as little memory as possible.
        gc.collect()

        # Continue evaluating the remaining tracks rather than
        # destroying all progress because of one problematic track.
        continue


# ----------------------------------------------------------------
# 12. RELOAD ALL SUCCESSFUL RESULTS
# ----------------------------------------------------------------

successful_results = []

if TRACK_RESULTS_JSONL.exists():

    with open(
        TRACK_RESULTS_JSONL,
        "r",
        encoding="utf-8",
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

                if record.get("status") == "success":
                    successful_results.append(record)

            except json.JSONDecodeError:
                continue


# Remove duplicate records by track, retaining the latest occurrence.
latest_by_track = {}

for record in successful_results:
    latest_by_track[record["track"]] = record

successful_results = list(
    latest_by_track.values()
)

successful_results.sort(
    key=lambda x: x["track"].lower()
)


# ----------------------------------------------------------------
# 13. WRITE FLAT CSV METRICS
# ----------------------------------------------------------------

import csv

csv_rows = []

for record in successful_results:

    track_name = record["track"]

    for source_name in SOURCE_ORDER:

        metrics = record["sources"][source_name]

        csv_rows.append({
            "track": track_name,
            "source": source_name,
            "SDR_dB": metrics["SDR_dB"],
            "ISR_dB": metrics["ISR_dB"],
            "SIR_dB": metrics["SIR_dB"],
            "SAR_dB": metrics["SAR_dB"],
            "SI_SDR_dB": metrics["SI_SDR_dB"],
        })


with open(
    TRACK_RESULTS_CSV,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "track",
            "source",
            "SDR_dB",
            "ISR_dB",
            "SIR_dB",
            "SAR_dB",
            "SI_SDR_dB",
        ],
    )

    writer.writeheader()
    writer.writerows(csv_rows)


# ----------------------------------------------------------------
# 14. FINAL SUMMARY
# ----------------------------------------------------------------

evaluation_elapsed = time.time() - evaluation_start

summary = {
    "evaluation": {
        "status": (
            "complete"
            if len(successful_results) == EXPECTED_TRACK_COUNT
            else "incomplete"
        ),
        "timestamp": datetime.now().isoformat(),
        "elapsed_seconds": round(evaluation_elapsed, 3),
        "expected_tracks": EXPECTED_TRACK_COUNT,
        "successful_tracks": len(successful_results),
        "failed_tracks_this_run": len(failed_results),
        "metrics": [
            "SDR",
            "ISR",
            "SIR",
            "SAR",
            "SI-SDR",
        ],
    },

    "bsseval": {
        "implementation": "museval.metrics.bss_eval",
        "version": "BSSEval v4",
        "window": "whole_track",
        "framewise_filters": False,
        "compute_permutation": False,
        "filters_len": BSSEVAL_FILTERS_LEN,
        "bsseval_sources_version": False,
    },

    "si_sdr": {
        "implementation": "standard scale-invariant projection",
        "zero_mean": False,
    },

    "paths": {
        "musdb18_test": str(MUSDB18_TEST_DIR),
        "predictions": str(PREDICTIONS_DIR),
        "metrics": str(METRICS_DIR),
        "plots": str(PLOTS_DIR),
        "reports": str(REPORTS_DIR),
        "logs": str(LOGS_DIR),
    },
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        allow_nan=True,
    )


# ----------------------------------------------------------------
# 15. FINAL STATUS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("OBJECTIVE EVALUATION RESULT")
print("=" * 70)

print(f"\nExpected tracks:   {EXPECTED_TRACK_COUNT}")
print(f"Successful tracks: {len(successful_results)}")
print(f"Failed this run:   {len(failed_results)}")

print("\nMetrics calculated:")
print("  ✓ BSSEval SDR")
print("  ✓ BSSEval ISR")
print("  ✓ BSSEval SIR")
print("  ✓ BSSEval SAR")
print("  ✓ SI-SDR")

print("\nOutput files:")
print(f"  Per-track JSONL : {TRACK_RESULTS_JSONL}")
print(f"  Flat metrics CSV: {TRACK_RESULTS_CSV}")
print(f"  Summary JSON     : {SUMMARY_JSON}")
print(f"  Evaluation log   : {LOG_FILE}")

if failed_results:
    print("\n⚠ Some tracks failed during this run:")

    for failed in failed_results:
        print(
            f"  - {failed['track']}: "
            f"{failed['error']}"
        )

if len(successful_results) == EXPECTED_TRACK_COUNT:

    print("\n" + "=" * 70)
    print("✓ CELL 9 COMPLETE")
    print("=" * 70)

    print(
        "\nAll 50 MUSDB18 test tracks have been evaluated."
    )

else:

    print("\n" + "=" * 70)
    print("⚠ CELL 9 INCOMPLETE")
    print("=" * 70)

    print(
        "\nNot all 50 tracks have successful results."
        "\nReview the evaluation log before proceeding."
    )


write_log(
    f"Cell 9 finished. "
    f"Successful tracks: {len(successful_results)}/"
    f"{EXPECTED_TRACK_COUNT}."
)

gc.collect()

OBJECTIVE EVALUATION — BSSEVAL v4 + SI-SDR

Python:
C:\IKS_Local_Runtime\iks_eval\Scripts\python.exe

Dependency status:
  museval: installed
  soundfile: installed
  numpy: installed
[2026-08-16 00:06:49] Cell 9 objective evaluation started.

EVALUATION PLAN

MUSDB18 test tracks: 50
Sources per track: 4
BSSEval: BSSEval v4
BSSEval framewise filters: False
BSSEval permutation: False
SI-SDR: separate calculation
Evaluation mode: one track at a time

----------------------------------------------------------------------
TRACK 1/50: Al James - Schoolboy Facination
----------------------------------------------------------------------
✓ Existing successful evaluation found.
  Skipping recalculation for this track.

----------------------------------------------------------------------
TRACK 2/50: AM Contra - Heart Peripheral
----------------------------------------------------------------------
✓ Existing successful evaluation found.
  Skipping recalculation for this track.

--------------

33

# Objective Metrics Inspection & Integrity Audit

In [10]:
# ================================================================
# CELL 10 — OBJECTIVE METRICS INSPECTION & INTEGRITY AUDIT
# ================================================================

import csv
import json
from pathlib import Path
from datetime import datetime


print("=" * 70)
print("OBJECTIVE METRICS INSPECTION & INTEGRITY AUDIT")
print("=" * 70)


# ----------------------------------------------------------------
# 1. VERIFY REQUIRED VARIABLES
# ----------------------------------------------------------------

required_variables = [
    "METRICS_DIR",
    "REPORTS_DIR",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Required configuration variables are missing:\n"
        + "\n".join(f"  - {name}" for name in missing_variables)
    )


METRICS_DIR = Path(METRICS_DIR)
REPORTS_DIR = Path(REPORTS_DIR)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)


# Derive the experiment name from the already validated directory.
# Example:
# ...\Objective\EXP002\metrics
#                    ↑
#                experiment
EXPERIMENT_NAME = METRICS_DIR.parent.name


print(f"\nExperiment: {EXPERIMENT_NAME}")
print(f"Started:    {datetime.now().isoformat()}")


# ----------------------------------------------------------------
# 2. METRIC POLICY
# ----------------------------------------------------------------

APPROVED_METRICS = [
    "SDR",
    "SIR",
    "SAR",
    "SI-SDR",
]

EXCLUDED_METRIC = "ISR"

EXPECTED_SOURCES = [
    "vocals",
    "drums",
    "bass",
    "other",
]

EXPECTED_TRACKS = 50

SOURCE_METRIC_KEYS = {
    "SDR": "SDR_dB",
    "SIR": "SIR_dB",
    "SAR": "SAR_dB",
    "SI-SDR": "SI_SDR_dB",
}

ISR_KEY = "ISR_dB"


print("\nApproved downstream metrics:")

for metric in APPROVED_METRICS:
    print(f"  ✓ {metric}")


print("\nExcluded metric:")
print(f"  ✗ {EXCLUDED_METRIC} (ignored downstream)")


print("\nResource policy:")
print("  ✓ No WAV files will be loaded")
print("  ✓ No objective metrics will be recomputed")
print("  ✓ JSONL inspected according to its actual schema")
print("  ✓ CSV inspected according to its actual schema")
print("  ✓ Track-level and source-level values are distinguished")
print("  ✓ RAM-safe validation")


# ----------------------------------------------------------------
# 3. OUTPUT FILES
# ----------------------------------------------------------------

TRACK_RESULTS_JSONL = (
    METRICS_DIR / "per_track_metrics.jsonl"
)

TRACK_RESULTS_CSV = (
    METRICS_DIR / "per_track_metrics.csv"
)

SUMMARY_JSON = (
    METRICS_DIR / "evaluation_summary.json"
)

AUDIT_REPORT = (
    REPORTS_DIR / "metrics_integrity_audit.json"
)


print("\n" + "-" * 70)
print("METRIC FILE AVAILABILITY")
print("-" * 70)


required_files = {
    "Per-track JSONL": TRACK_RESULTS_JSONL,
    "Per-track CSV": TRACK_RESULTS_CSV,
    "Evaluation summary": SUMMARY_JSON,
}


for label, path in required_files.items():

    if path.exists():
        print(f"✓ {label}: {path}")

    else:
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )


# ----------------------------------------------------------------
# 4. JSONL STRUCTURAL VALIDATION
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("JSONL STRUCTURAL VALIDATION")
print("-" * 70)


jsonl_records = []
jsonl_errors = []


with open(
    TRACK_RESULTS_JSONL,
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(f, start=1):

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)
            jsonl_records.append(record)

        except json.JSONDecodeError as exc:

            jsonl_errors.append(
                {
                    "line": line_number,
                    "error": str(exc),
                }
            )


if jsonl_errors:

    raise ValueError(
        f"JSONL contains {len(jsonl_errors)} invalid JSON record(s)."
    )


print(
    f"JSONL records discovered: {len(jsonl_records)}"
)


if len(jsonl_records) != EXPECTED_TRACKS:

    raise ValueError(
        f"Expected {EXPECTED_TRACKS} track-level JSONL records, "
        f"but found {len(jsonl_records)}."
    )


print("✓ Correct track-level record count")


# ----------------------------------------------------------------
# 5. TRACK VALIDATION
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK VALIDATION")
print("-" * 70)


tracks = []


for record in jsonl_records:

    track = record.get("track")

    if not isinstance(track, str) or not track.strip():

        raise ValueError(
            "A JSONL record is missing a valid 'track' identifier."
        )

    tracks.append(track)


unique_tracks = set(tracks)


print(
    f"Track identifiers discovered: {len(tracks)}"
)

print(
    f"Unique tracks discovered:     {len(unique_tracks)}"
)


if len(unique_tracks) != EXPECTED_TRACKS:

    raise ValueError(
        f"Expected {EXPECTED_TRACKS} unique tracks, "
        f"but found {len(unique_tracks)}."
    )


print(f"✓ {EXPECTED_TRACKS} unique tracks")


# ----------------------------------------------------------------
# 6. JSONL SOURCE-LEVEL STRUCTURE
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("JSONL SOURCE STRUCTURE")
print("-" * 70)


source_counts = {
    source: 0
    for source in EXPECTED_SOURCES
}


metric_counts = {
    metric: 0
    for metric in APPROVED_METRICS
}


isr_source_count = 0


track_average_counts = {
    metric: 0
    for metric in APPROVED_METRICS
}


track_average_isr_count = 0


structural_errors = []


for record in jsonl_records:

    track = record["track"]

    sources = record.get("sources")


    if not isinstance(sources, dict):

        structural_errors.append(
            f"{track}: missing or invalid 'sources' object."
        )

        continue


    for source in EXPECTED_SOURCES:

        if source not in sources:

            structural_errors.append(
                f"{track}: missing source '{source}'."
            )

            continue


        source_counts[source] += 1

        source_metrics = sources[source]


        if not isinstance(source_metrics, dict):

            structural_errors.append(
                f"{track}/{source}: "
                "source metrics are not an object."
            )

            continue


        # --------------------------------------------------------
        # Approved metrics
        # --------------------------------------------------------

        for metric, key in SOURCE_METRIC_KEYS.items():

            if key not in source_metrics:

                structural_errors.append(
                    f"{track}/{source}: missing {metric}."
                )

                continue


            value = source_metrics[key]


            if not isinstance(value, (int, float)):

                structural_errors.append(
                    f"{track}/{source}: "
                    f"{metric} is not numeric."
                )

                continue


            metric_counts[metric] += 1


        # --------------------------------------------------------
        # ISR
        #
        # ISR may exist in EXP002 because Cell 9 calculated it.
        # It is only counted for auditing and is NOT approved.
        # --------------------------------------------------------

        if ISR_KEY in source_metrics:

            value = source_metrics[ISR_KEY]

            if isinstance(value, (int, float)):

                isr_source_count += 1


# ----------------------------------------------------------------
# 7. SOURCE COVERAGE
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("SOURCE COVERAGE")
print("-" * 70)


for source in EXPECTED_SOURCES:

    print(
        f"{source:>6}: "
        f"{source_counts[source]}/{EXPECTED_TRACKS}"
    )

    if source_counts[source] != EXPECTED_TRACKS:

        structural_errors.append(
            f"Source '{source}' does not occur in all "
            f"{EXPECTED_TRACKS} tracks."
        )


if structural_errors:

    print("\n✗ Structural errors detected:")

    for error in structural_errors[:30]:
        print(f"  - {error}")


    if len(structural_errors) > 30:

        print(
            f"  ... {len(structural_errors) - 30} "
            "additional errors omitted."
        )


    raise ValueError(
        f"JSONL structural validation failed with "
        f"{len(structural_errors)} error(s)."
    )


print(
    "✓ All tracks contain all four expected sources"
)


# ----------------------------------------------------------------
# 8. APPROVED SOURCE-LEVEL METRIC VALIDATION
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("APPROVED SOURCE-LEVEL METRIC VALIDATION")
print("-" * 70)


EXPECTED_SOURCE_EVALUATIONS = (
    EXPECTED_TRACKS * len(EXPECTED_SOURCES)
)


EXPECTED_APPROVED_VALUES = (
    EXPECTED_SOURCE_EVALUATIONS
    * len(APPROVED_METRICS)
)


print(
    f"Expected source evaluations: "
    f"{EXPECTED_SOURCE_EVALUATIONS}"
)


for metric in APPROVED_METRICS:

    expected = EXPECTED_SOURCE_EVALUATIONS

    actual = metric_counts[metric]


    print(
        f"  {metric:>6}: "
        f"{actual}/{expected}"
    )


    if actual != expected:

        raise ValueError(
            f"{metric}: expected {expected} source-level "
            f"values but found {actual}."
        )


    print(
        f"       ✓ {metric} present and numerically valid"
    )


actual_approved_values = sum(
    metric_counts.values()
)


print(
    f"\nApproved source-level metric values: "
    f"{actual_approved_values}"
)


if actual_approved_values != EXPECTED_APPROVED_VALUES:

    raise ValueError(
        f"Expected {EXPECTED_APPROVED_VALUES} approved "
        f"source-level metric values, but found "
        f"{actual_approved_values}."
    )


print(
    "✓ Correct approved source-level metric coverage"
)


# ----------------------------------------------------------------
# 9. TRACK-AVERAGE STRUCTURE
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK-AVERAGE STRUCTURE")
print("-" * 70)


for record in jsonl_records:

    track = record["track"]

    track_average = record.get("track_average")


    if not isinstance(track_average, dict):

        raise ValueError(
            f"{track}: missing or invalid "
            "track_average object."
        )


    for metric, key in SOURCE_METRIC_KEYS.items():

        if key not in track_average:

            raise ValueError(
                f"{track}: track_average missing {metric}."
            )


        value = track_average[key]


        if value is not None and not isinstance(
            value,
            (int, float),
        ):

            raise ValueError(
                f"{track}: track_average {metric} "
                "is not numeric."
            )


        if value is not None:

            track_average_counts[metric] += 1


    if ISR_KEY in track_average:

        value = track_average[ISR_KEY]


        if value is not None and isinstance(
            value,
            (int, float),
        ):

            track_average_isr_count += 1


for metric in APPROVED_METRICS:

    print(
        f"  {metric:>6}: "
        f"{track_average_counts[metric]}/"
        f"{EXPECTED_TRACKS}"
    )


# ----------------------------------------------------------------
# 10. EXCLUDED ISR POLICY
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("EXCLUDED ISR POLICY")
print("-" * 70)


print(
    f"✓ Raw EXP002 source-level ISR values discovered: "
    f"{isr_source_count}"
)


print(
    f"✓ Raw EXP002 track-average ISR values discovered: "
    f"{track_average_isr_count}"
)


print("✓ ISR is excluded from downstream analysis.")
print("✓ ISR will not enter downstream reports.")
print("✓ ISR will not enter downstream plots.")
print("✓ ISR will not enter downstream summary tables.")
print("✓ ISR will NOT be calculated for EXP003.")
print("✓ ISR will NOT be calculated for EXP004.")


# ----------------------------------------------------------------
# 11. CSV STRUCTURAL VALIDATION
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("CSV STRUCTURAL VALIDATION")
print("-" * 70)


with open(
    TRACK_RESULTS_CSV,
    "r",
    newline="",
    encoding="utf-8",
) as f:

    reader = csv.DictReader(f)

    csv_columns = reader.fieldnames or []

    csv_rows = list(reader)


print(
    f"CSV columns discovered: {len(csv_columns)}"
)


for column in csv_columns:

    print(f"  - {column}")


EXPECTED_CSV_COLUMNS = [
    "track",
    "source",
    "SDR_dB",
    "ISR_dB",
    "SIR_dB",
    "SAR_dB",
    "SI_SDR_dB",
]


missing_csv_columns = [
    column
    for column in EXPECTED_CSV_COLUMNS
    if column not in csv_columns
]


if missing_csv_columns:

    raise ValueError(
        "CSV is missing expected columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_csv_columns
        )
    )


print(
    f"\nCSV data rows discovered: {len(csv_rows)}"
)


if len(csv_rows) != EXPECTED_SOURCE_EVALUATIONS:

    raise ValueError(
        f"Expected {EXPECTED_SOURCE_EVALUATIONS} "
        f"source-level CSV rows, but found "
        f"{len(csv_rows)}."
    )


print(
    f"✓ {EXPECTED_TRACKS} tracks × "
    f"{len(EXPECTED_SOURCES)} sources = "
    f"{EXPECTED_SOURCE_EVALUATIONS} CSV rows"
)


# ----------------------------------------------------------------
# 12. CSV SOURCE AND METRIC COVERAGE
# ----------------------------------------------------------------

csv_source_counts = {
    source: 0
    for source in EXPECTED_SOURCES
}


csv_metric_counts = {
    metric: 0
    for metric in APPROVED_METRICS
}


csv_isr_count = 0


for row in csv_rows:

    source = row["source"]


    if source not in EXPECTED_SOURCES:

        raise ValueError(
            f"Unexpected CSV source: {source}"
        )


    csv_source_counts[source] += 1


    for metric, key in SOURCE_METRIC_KEYS.items():

        value = row[key]


        try:

            numeric_value = float(value)

        except (TypeError, ValueError):

            raise ValueError(
                f"Invalid CSV value for {metric}: "
                f"{value}"
            )


        if numeric_value != numeric_value:

            raise ValueError(
                f"NaN CSV value found for {metric}."
            )


        csv_metric_counts[metric] += 1


    # ISR is inspected only as legacy/raw data.
    if ISR_KEY in row:

        try:

            float(row[ISR_KEY])

            csv_isr_count += 1

        except (TypeError, ValueError):

            pass


print("\nCSV source coverage:")


for source in EXPECTED_SOURCES:

    print(
        f"  {source:>6}: "
        f"{csv_source_counts[source]}/"
        f"{EXPECTED_TRACKS}"
    )


    if csv_source_counts[source] != EXPECTED_TRACKS:

        raise ValueError(
            f"CSV source coverage error for {source}."
        )


print("\nCSV approved metric coverage:")


for metric in APPROVED_METRICS:

    print(
        f"  {metric:>6}: "
        f"{csv_metric_counts[metric]}"
    )


    if (
        csv_metric_counts[metric]
        != EXPECTED_SOURCE_EVALUATIONS
    ):

        raise ValueError(
            f"CSV {metric} coverage is incorrect."
        )


print(
    f"\n✓ Raw EXP002 CSV ISR values discovered: "
    f"{csv_isr_count}"
)


print(
    "✓ CSV approved metrics are numerically valid"
)


# ----------------------------------------------------------------
# 13. CROSS-FILE TRACK VALIDATION
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("CROSS-FILE TRACK VALIDATION")
print("-" * 70)


csv_tracks = {
    row["track"]
    for row in csv_rows
}


jsonl_tracks = set(unique_tracks)


print(
    f"JSONL tracks: {len(jsonl_tracks)}"
)


print(
    f"CSV tracks:   {len(csv_tracks)}"
)


if jsonl_tracks != csv_tracks:

    missing_from_csv = (
        jsonl_tracks - csv_tracks
    )

    missing_from_jsonl = (
        csv_tracks - jsonl_tracks
    )


    raise ValueError(
        "JSONL/CSV track mismatch.\n"
        f"Missing from CSV: "
        f"{sorted(missing_from_csv)}\n"
        f"Missing from JSONL: "
        f"{sorted(missing_from_jsonl)}"
    )


print(
    "✓ JSONL and CSV contain the same tracks"
)


# ----------------------------------------------------------------
# 14. EVALUATION SUMMARY VALIDATION
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("EVALUATION SUMMARY VALIDATION")
print("-" * 70)


with open(
    SUMMARY_JSON,
    "r",
    encoding="utf-8",
) as f:

    summary = json.load(f)


if not isinstance(summary, dict):

    raise ValueError(
        "Evaluation summary is not a JSON object."
    )


print("✓ Evaluation summary is valid JSON")


print("\nSummary keys discovered:")


for key in summary.keys():

    print(f"  - {key}")


# ----------------------------------------------------------------
# 15. WRITE AUDIT REPORT
# ----------------------------------------------------------------

audit_report = {

    "experiment": EXPERIMENT_NAME,

    "timestamp": datetime.now().isoformat(),

    "expected_tracks": EXPECTED_TRACKS,

    "expected_sources": EXPECTED_SOURCES,

    "approved_metrics": APPROVED_METRICS,

    "excluded_metric": EXCLUDED_METRIC,

    "jsonl": {

        "records": len(jsonl_records),

        "unique_tracks": len(unique_tracks),

        "source_evaluations":
            EXPECTED_SOURCE_EVALUATIONS,

        "approved_source_metric_values": {
            metric: metric_counts[metric]
            for metric in APPROVED_METRICS
        },

        "excluded_isr_source_values":
            isr_source_count,

        "track_average_values": {
            metric: track_average_counts[metric]
            for metric in APPROVED_METRICS
        },

        "track_average_isr_values":
            track_average_isr_count,
    },

    "csv": {

        "data_rows": len(csv_rows),

        "source_evaluations":
            EXPECTED_SOURCE_EVALUATIONS,

        "approved_metric_values": {
            metric: csv_metric_counts[metric]
            for metric in APPROVED_METRICS
        },

        "isr_values":
            csv_isr_count,
    },

    "policy": {

        "isr_included_downstream": False,

        "isr_included_in_reports": False,

        "isr_included_in_plots": False,

        "isr_included_in_summary_tables": False,

        "isr_calculated_in_exp003": False,

        "isr_calculated_in_exp004": False,
    },

    "files": {

        "jsonl":
            str(TRACK_RESULTS_JSONL),

        "csv":
            str(TRACK_RESULTS_CSV),

        "summary":
            str(SUMMARY_JSON),

        "audit":
            str(AUDIT_REPORT),
    },
}


with open(
    AUDIT_REPORT,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        audit_report,
        f,
        indent=2,
    )


# ----------------------------------------------------------------
# 16. FINAL RESULT
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 10 AUDIT RESULT")
print("=" * 70)


print(
    f"\nJSONL records:             "
    f"{len(jsonl_records)}"
)


print(
    f"Unique JSONL tracks:       "
    f"{len(unique_tracks)}"
)


print(
    f"Source evaluations:        "
    f"{EXPECTED_SOURCE_EVALUATIONS}"
)


print("\nApproved source-level metrics:")


for metric in APPROVED_METRICS:

    print(
        f"  ✓ {metric}: "
        f"{metric_counts[metric]}"
    )


print(
    f"\nRaw ISR source values:     "
    f"{isr_source_count}"
)


print(
    f"Raw ISR track averages:    "
    f"{track_average_isr_count}"
)


print(
    f"\nCSV source evaluations:    "
    f"{len(csv_rows)}"
)


print(
    f"CSV columns:               "
    f"{len(csv_columns)}"
)


print(
    f"\nAudit report: {AUDIT_REPORT}"
)


print("\n" + "=" * 70)
print("✓ CELL 10 COMPLETE")
print("=" * 70)


print(
    "\nEXP002 metrics were inspected without loading WAV files "
    "or recomputing any metrics."
)


print(
    "ISR was identified as a legacy/raw metric and excluded "
    "from downstream analysis."
)

OBJECTIVE METRICS INSPECTION & INTEGRITY AUDIT

Experiment: EXP002
Started:    2026-08-16T00:07:11.468788

Approved downstream metrics:
  ✓ SDR
  ✓ SIR
  ✓ SAR
  ✓ SI-SDR

Excluded metric:
  ✗ ISR (ignored downstream)

Resource policy:
  ✓ No WAV files will be loaded
  ✓ No objective metrics will be recomputed
  ✓ JSONL inspected according to its actual schema
  ✓ CSV inspected according to its actual schema
  ✓ Track-level and source-level values are distinguished
  ✓ RAM-safe validation

----------------------------------------------------------------------
METRIC FILE AVAILABILITY
----------------------------------------------------------------------
✓ Per-track JSONL: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics\per_track_metrics.jsonl
✓ Per-track CSV: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics\per_track_metrics.csv
✓ Evaluation summary: G:

# Cell 11 — EXP002 Metric Descriptive Statistics

In [11]:
# ================================================================
# CELL 11 — EXP002 METRIC DESCRIPTIVE STATISTICS
# ================================================================

import csv
import json
import math
from pathlib import Path

import numpy as np


# ----------------------------------------------------------------
# 1. CELL HEADER
# ----------------------------------------------------------------

print("=" * 70)
print("EXP002 METRIC DESCRIPTIVE STATISTICS")
print("=" * 70)


# ----------------------------------------------------------------
# 2. CONFIGURATION DISCOVERY
# ----------------------------------------------------------------

# Cell 10 should already have established these variables.
# We deliberately use the existing experiment configuration rather
# than hard-coding a new location.

required_variables = [
    "METRICS_DIR",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Required configuration variables are missing:\n"
        + "\n".join(f"  - {name}" for name in missing_variables)
    )


METRICS_DIR = Path(METRICS_DIR)

CSV_PATH = METRICS_DIR / "per_track_metrics.csv"
JSONL_PATH = METRICS_DIR / "per_track_metrics.jsonl"


APPROVED_METRICS = [
    "SDR",
    "SIR",
    "SAR",
    "SI-SDR",
]

METRIC_COLUMNS = {
    "SDR": "SDR_dB",
    "SIR": "SIR_dB",
    "SAR": "SAR_dB",
    "SI-SDR": "SI_SDR_dB",
}

EXPECTED_SOURCES = [
    "vocals",
    "drums",
    "bass",
    "other",
]

EXPECTED_TRACKS = 50
EXPECTED_SOURCE_EVALUATIONS = EXPECTED_TRACKS * len(EXPECTED_SOURCES)


# ----------------------------------------------------------------
# 3. INPUT VALIDATION
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("INPUT VALIDATION")
print("-" * 70)

print(f"Metrics directory: {METRICS_DIR}")

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Required CSV file was not found:\n{CSV_PATH}"
    )

if not JSONL_PATH.exists():
    raise FileNotFoundError(
        f"Required JSONL file was not found:\n{JSONL_PATH}"
    )

print(f"✓ CSV found:   {CSV_PATH}")
print(f"✓ JSONL found:  {JSONL_PATH}")


# ----------------------------------------------------------------
# 4. LOAD SOURCE-LEVEL CSV DATA
# ----------------------------------------------------------------
#
# The CSV is the source-level flat representation:
#
#     50 tracks × 4 sources = 200 rows
#
# We use it for source-level descriptive statistics.
#
# ISR is intentionally not read into the analysis.
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("SOURCE-LEVEL DATA DISCOVERY")
print("-" * 70)

source_rows = []

with open(
    CSV_PATH,
    "r",
    newline="",
    encoding="utf-8",
) as f:

    reader = csv.DictReader(f)

    csv_columns = reader.fieldnames or []

    required_csv_columns = [
        "track",
        "source",
        *METRIC_COLUMNS.values(),
    ]

    missing_csv_columns = [
        column
        for column in required_csv_columns
        if column not in csv_columns
    ]

    if missing_csv_columns:
        raise ValueError(
            "Required CSV columns are missing:\n"
            + "\n".join(
                f"  - {column}"
                for column in missing_csv_columns
            )
        )

    for row_number, row in enumerate(reader, start=2):

        track = row.get("track", "").strip()
        source = row.get("source", "").strip()

        if not track:
            raise ValueError(
                f"CSV row {row_number}: missing track identifier."
            )

        if source not in EXPECTED_SOURCES:
            raise ValueError(
                f"CSV row {row_number}: unexpected source "
                f"'{source}'."
            )

        parsed_row = {
            "track": track,
            "source": source,
        }

        for metric_name, column_name in METRIC_COLUMNS.items():

            raw_value = row.get(column_name)

            try:
                value = float(raw_value)
            except (TypeError, ValueError):
                raise ValueError(
                    f"CSV row {row_number}: invalid {metric_name} "
                    f"value '{raw_value}'."
                )

            if not math.isfinite(value):
                raise ValueError(
                    f"CSV row {row_number}: non-finite {metric_name} "
                    f"value '{raw_value}'."
                )

            parsed_row[metric_name] = value

        source_rows.append(parsed_row)


print(f"CSV source-level rows discovered: {len(source_rows)}")

if len(source_rows) != EXPECTED_SOURCE_EVALUATIONS:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_EVALUATIONS} source-level rows, "
        f"but discovered {len(source_rows)}."
    )

print(
    f"✓ Correct source-level row count: "
    f"{EXPECTED_TRACKS} tracks × "
    f"{len(EXPECTED_SOURCES)} sources"
)


# ----------------------------------------------------------------
# 5. SOURCE / TRACK COVERAGE
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("SOURCE / TRACK COVERAGE")
print("-" * 70)

tracks = sorted(
    {
        row["track"]
        for row in source_rows
    },
    key=str.lower,
)

print(f"Unique tracks: {len(tracks)}")

if len(tracks) != EXPECTED_TRACKS:
    raise ValueError(
        f"Expected {EXPECTED_TRACKS} unique tracks, "
        f"but found {len(tracks)}."
    )

for source in EXPECTED_SOURCES:

    source_tracks = {
        row["track"]
        for row in source_rows
        if row["source"] == source
    }

    print(
        f"{source:>6}: "
        f"{len(source_tracks)}/{EXPECTED_TRACKS}"
    )

    if source_tracks != set(tracks):
        raise ValueError(
            f"Track coverage mismatch for source '{source}'."
        )

print("✓ Complete source coverage confirmed")


# ----------------------------------------------------------------
# 6. DESCRIPTIVE STATISTICS FUNCTION
# ----------------------------------------------------------------

def descriptive_statistics(values):
    """
    Calculate descriptive statistics for a finite numeric array.
    """

    values = np.asarray(values, dtype=np.float64)

    if values.size == 0:
        return {
            "count": 0,
            "mean_dB": None,
            "median_dB": None,
            "std_dB": None,
            "min_dB": None,
            "q1_dB": None,
            "q3_dB": None,
            "max_dB": None,
            "iqr_dB": None,
        }

    q1 = float(np.percentile(values, 25))
    q3 = float(np.percentile(values, 75))

    return {
        "count": int(values.size),
        "mean_dB": float(np.mean(values)),
        "median_dB": float(np.median(values)),
        "std_dB": float(np.std(values, ddof=1))
            if values.size > 1
            else 0.0,
        "min_dB": float(np.min(values)),
        "q1_dB": q1,
        "q3_dB": q3,
        "max_dB": float(np.max(values)),
        "iqr_dB": q3 - q1,
    }


# ----------------------------------------------------------------
# 7. OVERALL METRIC STATISTICS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL METRIC STATISTICS")
print("=" * 70)

overall_statistics = {}

for metric in APPROVED_METRICS:

    values = [
        row[metric]
        for row in source_rows
    ]

    stats = descriptive_statistics(values)

    overall_statistics[metric] = stats

    print(f"\n{metric}")
    print("-" * 40)
    print(f"  Count:   {stats['count']}")
    print(f"  Mean:    {stats['mean_dB']:.3f} dB")
    print(f"  Median:  {stats['median_dB']:.3f} dB")
    print(f"  Std:     {stats['std_dB']:.3f} dB")
    print(f"  Min:     {stats['min_dB']:.3f} dB")
    print(f"  Q1:      {stats['q1_dB']:.3f} dB")
    print(f"  Q3:      {stats['q3_dB']:.3f} dB")
    print(f"  Max:     {stats['max_dB']:.3f} dB")
    print(f"  IQR:     {stats['iqr_dB']:.3f} dB")


# ----------------------------------------------------------------
# 8. SOURCE-BY-SOURCE STATISTICS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE-BY-SOURCE METRIC STATISTICS")
print("=" * 70)

source_statistics = {}

for source in EXPECTED_SOURCES:

    source_statistics[source] = {}

    print(f"\n{source.upper()}")
    print("-" * 70)

    print(
        f"{'Metric':<10}"
        f"{'Mean':>12}"
        f"{'Median':>12}"
        f"{'Std':>12}"
        f"{'Min':>12}"
        f"{'Q1':>12}"
        f"{'Q3':>12}"
        f"{'Max':>12}"
    )

    print("-" * 70)

    source_subset = [
        row
        for row in source_rows
        if row["source"] == source
    ]

    for metric in APPROVED_METRICS:

        values = [
            row[metric]
            for row in source_subset
        ]

        stats = descriptive_statistics(values)

        source_statistics[source][metric] = stats

        print(
            f"{metric:<10}"
            f"{stats['mean_dB']:>12.3f}"
            f"{stats['median_dB']:>12.3f}"
            f"{stats['std_dB']:>12.3f}"
            f"{stats['min_dB']:>12.3f}"
            f"{stats['q1_dB']:>12.3f}"
            f"{stats['q3_dB']:>12.3f}"
            f"{stats['max_dB']:>12.3f}"
        )


# ----------------------------------------------------------------
# 9. SOURCE MEAN SUMMARY
# ----------------------------------------------------------------
#
# This is intentionally a compact summary for the next analysis cell.
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE MEAN SUMMARY")
print("=" * 70)

print(
    f"\n{'Source':<12}"
    f"{'SDR':>12}"
    f"{'SIR':>12}"
    f"{'SAR':>12}"
    f"{'SI-SDR':>12}"
)

print("-" * 60)

for source in EXPECTED_SOURCES:

    print(
        f"{source:<12}"
        f"{source_statistics[source]['SDR']['mean_dB']:>12.3f}"
        f"{source_statistics[source]['SIR']['mean_dB']:>12.3f}"
        f"{source_statistics[source]['SAR']['mean_dB']:>12.3f}"
        f"{source_statistics[source]['SI-SDR']['mean_dB']:>12.3f}"
    )


# ----------------------------------------------------------------
# 10. LOAD TRACK-AVERAGE VALUES FROM JSONL
# ----------------------------------------------------------------
#
# IMPORTANT:
#
# Track averages are taken directly from Cell 9's stored
# `track_average` objects.
#
# We do NOT reconstruct track averages from the CSV here.
# This preserves the exact aggregation performed during evaluation.
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK-AVERAGE DISCOVERY")
print("-" * 70)

track_average_records = {}

with open(
    JSONL_PATH,
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(f, start=1):

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue

        if record.get("status") != "success":
            continue

        track = record.get("track")

        if not track:
            continue

        track_average = record.get("track_average")

        if not isinstance(track_average, dict):
            raise ValueError(
                f"Track '{track}' has no valid track_average object."
            )

        parsed_average = {}

        for metric in APPROVED_METRICS:

            column_name = METRIC_COLUMNS[metric]

            if column_name not in track_average:
                raise ValueError(
                    f"Track '{track}' is missing "
                    f"track-average metric '{column_name}'."
                )

            value = track_average[column_name]

            if value is None:
                raise ValueError(
                    f"Track '{track}' has null "
                    f"track-average '{column_name}'."
                )

            value = float(value)

            if not math.isfinite(value):
                raise ValueError(
                    f"Track '{track}' has non-finite "
                    f"track-average '{column_name}'."
                )

            parsed_average[metric] = value

        track_average_records[track] = parsed_average


print(
    f"Track-average records discovered: "
    f"{len(track_average_records)}"
)

if len(track_average_records) != EXPECTED_TRACKS:
    raise ValueError(
        f"Expected {EXPECTED_TRACKS} track-average records, "
        f"but found {len(track_average_records)}."
    )

print("✓ All 50 stored track-average records are valid")


# ----------------------------------------------------------------
# 11. TRACK-AVERAGE STATISTICS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("TRACK-AVERAGE METRIC STATISTICS")
print("=" * 70)

track_average_statistics = {}

for metric in APPROVED_METRICS:

    values = [
        record[metric]
        for record in track_average_records.values()
    ]

    stats = descriptive_statistics(values)

    track_average_statistics[metric] = stats

    print(f"\n{metric}")
    print("-" * 40)
    print(f"  Count:   {stats['count']}")
    print(f"  Mean:    {stats['mean_dB']:.3f} dB")
    print(f"  Median:  {stats['median_dB']:.3f} dB")
    print(f"  Std:     {stats['std_dB']:.3f} dB")
    print(f"  Min:     {stats['min_dB']:.3f} dB")
    print(f"  Q1:      {stats['q1_dB']:.3f} dB")
    print(f"  Q3:      {stats['q3_dB']:.3f} dB")
    print(f"  Max:     {stats['max_dB']:.3f} dB")
    print(f"  IQR:     {stats['iqr_dB']:.3f} dB")


# ----------------------------------------------------------------
# 12. SAVE MACHINE-READABLE STATISTICS
# ----------------------------------------------------------------

statistics_report = {
    "experiment": "EXP002",
    "approved_metrics": APPROVED_METRICS,
    "excluded_metrics": [
        "ISR",
    ],
    "dataset": {
        "tracks": EXPECTED_TRACKS,
        "sources_per_track": len(EXPECTED_SOURCES),
        "source_evaluations": EXPECTED_SOURCE_EVALUATIONS,
    },
    "source_level_statistics": source_statistics,
    "overall_source_level_statistics": overall_statistics,
    "track_average_statistics": track_average_statistics,
}


REPORTS_DIR = Path(
    globals().get(
        "REPORTS_DIR",
        METRICS_DIR.parent / "reports",
    )
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

STATISTICS_JSON = (
    REPORTS_DIR /
    "exp002_metric_descriptive_statistics.json"
)

with open(
    STATISTICS_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        statistics_report,
        f,
        indent=2,
        allow_nan=False,
    )


# ----------------------------------------------------------------
# 13. FINAL STATUS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 11 RESULT")
print("=" * 70)

print(f"\nTracks analysed:          {EXPECTED_TRACKS}")
print(f"Source evaluations:      {EXPECTED_SOURCE_EVALUATIONS}")
print(f"Approved metrics:        {len(APPROVED_METRICS)}")

print("\nApproved metrics analysed:")
for metric in APPROVED_METRICS:
    print(f"  ✓ {metric}")

print("\nExcluded:")
print("  ✗ ISR")

print(f"\nStatistics report:")
print(f"  {STATISTICS_JSON}")

print("\n" + "=" * 70)
print("✓ CELL 11 COMPLETE")
print("=" * 70)

print(
    "\nEXP002 descriptive statistics were calculated from the existing "
    "evaluation outputs. No WAV files were loaded and no metrics were "
    "recomputed."
)

EXP002 METRIC DESCRIPTIVE STATISTICS

----------------------------------------------------------------------
INPUT VALIDATION
----------------------------------------------------------------------
Metrics directory: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics
✓ CSV found:   G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics\per_track_metrics.csv
✓ JSONL found:  G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics\per_track_metrics.jsonl

----------------------------------------------------------------------
SOURCE-LEVEL DATA DISCOVERY
----------------------------------------------------------------------
CSV source-level rows discovered: 200
✓ Correct source-level row count: 50 tracks × 4 sources

----------------------------------------------------------------------
SOURCE / TRACK COVERAG

In [ ]:
%pip install matplotlib

   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.3 MB 6.7 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.3 MB 4.8 MB/s eta 0:00:02
   ----------- ---------------------------- 2.6/9.3 MB 4.6 MB/s eta 0:00:02
   -------------- ------------------------- 3.4/9.3 MB 4.5 MB/s eta 0:00:02
   ------------------- -------------------- 4.5/9.3 MB 4.5 MB/s eta 0:00:02
   ---------------------- ----------------- 5.2/9.3 MB 4.5 MB/s eta 0:00:01
   ------------------------- -------------- 6.0/9.3 MB 4.3 MB/s eta 0:00:01
   ------------------------------ --------- 7.1/9.3 MB 4.4 MB/s eta 0:00:01
   -------------------------------- ------- 7.6/9.3 MB 4.4 MB/s eta 0:00:01
   ------------------------------------- -- 8.7/9.3 MB 4.3 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 4.2 MB/s  0:00:02
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ------------- ------------

# Cell 12 — Objective Metric Distribution and Outlier Analysis

In [13]:
# ================================================================
# CELL 12 — OBJECTIVE METRIC DISTRIBUTION & OUTLIER ANALYSIS
# EXP002
# ================================================================

import csv
import json
import math
from pathlib import Path
from datetime import datetime

import numpy as np


# ----------------------------------------------------------------
# 1. INITIALIZATION
# ----------------------------------------------------------------

print("=" * 70)
print("EXP002 OBJECTIVE METRIC DISTRIBUTION & OUTLIER ANALYSIS")
print("=" * 70)

START_TIME = datetime.now().isoformat()

# Use variables from previous cells when available.
# Cell 12 does not depend on a separately defined EXPERIMENT_NAME.
EXPERIMENT_NAME = globals().get("EXPERIMENT_NAME", "EXP002")

if "METRICS_DIR" not in globals():
    raise RuntimeError(
        "METRICS_DIR is not available. Run the previous configuration "
        "cells before running Cell 12."
    )

if "REPORTS_DIR" not in globals():
    raise RuntimeError(
        "REPORTS_DIR is not available. Run the previous configuration "
        "cells before running Cell 12."
    )

if "PLOTS_DIR" not in globals():
    raise RuntimeError(
        "PLOTS_DIR is not available. Run the previous configuration "
        "cells before running Cell 12."
    )

METRICS_DIR = Path(METRICS_DIR)
REPORTS_DIR = Path(REPORTS_DIR)
PLOTS_DIR = Path(PLOTS_DIR)

TRACK_RESULTS_CSV = METRICS_DIR / "per_track_metrics.csv"
TRACK_RESULTS_JSONL = METRICS_DIR / "per_track_metrics.jsonl"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

APPROVED_METRICS = [
    "SDR",
    "SIR",
    "SAR",
    "SI-SDR",
]

METRIC_COLUMNS = {
    "SDR": "SDR_dB",
    "SIR": "SIR_dB",
    "SAR": "SAR_dB",
    "SI-SDR": "SI_SDR_dB",
}

EXPECTED_SOURCES = [
    "vocals",
    "drums",
    "bass",
    "other",
]

EXPECTED_TRACKS = 50


# ----------------------------------------------------------------
# 2. INPUT VALIDATION
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("INPUT VALIDATION")
print("-" * 70)

print(f"Metrics directory: {METRICS_DIR}")
print(f"Reports directory: {REPORTS_DIR}")
print(f"Plots directory:   {PLOTS_DIR}")

if not TRACK_RESULTS_CSV.exists():
    raise FileNotFoundError(
        f"Required CSV not found: {TRACK_RESULTS_CSV}"
    )

if not TRACK_RESULTS_JSONL.exists():
    raise FileNotFoundError(
        f"Required JSONL not found: {TRACK_RESULTS_JSONL}"
    )

print("✓ Required metric files found")


# ----------------------------------------------------------------
# 3. LOAD FLAT SOURCE-LEVEL CSV
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("SOURCE-LEVEL DATA DISCOVERY")
print("-" * 70)

rows = []

with open(
    TRACK_RESULTS_CSV,
    "r",
    encoding="utf-8",
    newline="",
) as f:

    reader = csv.DictReader(f)

    csv_columns = reader.fieldnames or []

    required_columns = [
        "track",
        "source",
        *METRIC_COLUMNS.values(),
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in csv_columns
    ]

    if missing_columns:
        raise ValueError(
            "CSV is missing required columns:\n"
            + "\n".join(f"  - {column}" for column in missing_columns)
        )

    for row in reader:
        rows.append(row)

print(f"CSV rows discovered: {len(rows)}")

expected_rows = EXPECTED_TRACKS * len(EXPECTED_SOURCES)

if len(rows) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows} source-level rows "
        f"({EXPECTED_TRACKS} tracks × {len(EXPECTED_SOURCES)} sources), "
        f"but found {len(rows)}."
    )

print(
    f"✓ Correct source-level row count: "
    f"{EXPECTED_TRACKS} tracks × {len(EXPECTED_SOURCES)} sources"
)


# ----------------------------------------------------------------
# 4. NORMALIZE DATA
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("DATA NORMALIZATION")
print("-" * 70)

normalized_rows = []

for row_index, row in enumerate(rows, start=2):

    track = str(row["track"]).strip()
    source = str(row["source"]).strip()

    if not track:
        raise ValueError(
            f"CSV row {row_index}: empty track identifier."
        )

    if source not in EXPECTED_SOURCES:
        raise ValueError(
            f"CSV row {row_index}: unexpected source '{source}'."
        )

    normalized = {
        "track": track,
        "source": source,
    }

    for metric_name, column_name in METRIC_COLUMNS.items():

        raw_value = row.get(column_name)

        try:
            value = float(raw_value)
        except (TypeError, ValueError):
            raise ValueError(
                f"CSV row {row_index}: invalid {metric_name} "
                f"value '{raw_value}'."
            )

        if not np.isfinite(value):
            raise ValueError(
                f"CSV row {row_index}: non-finite {metric_name} value."
            )

        normalized[metric_name] = value

    normalized_rows.append(normalized)

print("✓ All approved metric values are numeric and finite")


# ----------------------------------------------------------------
# 5. TRACK / SOURCE COVERAGE
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK / SOURCE COVERAGE")
print("-" * 70)

tracks = sorted(
    {row["track"] for row in normalized_rows},
    key=str.lower,
)

if len(tracks) != EXPECTED_TRACKS:
    raise ValueError(
        f"Expected {EXPECTED_TRACKS} unique tracks, "
        f"but found {len(tracks)}."
    )

print(f"Unique tracks: {len(tracks)}")

for source in EXPECTED_SOURCES:

    source_tracks = {
        row["track"]
        for row in normalized_rows
        if row["source"] == source
    }

    print(
        f"{source:>7}: "
        f"{len(source_tracks)}/{EXPECTED_TRACKS}"
    )

    if len(source_tracks) != EXPECTED_TRACKS:
        raise ValueError(
            f"Incomplete track coverage for source '{source}'."
        )

print("✓ Complete source coverage confirmed")


# ----------------------------------------------------------------
# 6. STATISTICAL HELPERS
# ----------------------------------------------------------------

def percentile(values, q):
    """Return percentile using NumPy."""
    return float(np.percentile(np.asarray(values, dtype=float), q))


def descriptive_statistics(values):
    """Calculate descriptive statistics for a numeric sequence."""

    values = np.asarray(values, dtype=float)

    return {
        "count": int(len(values)),
        "mean_dB": float(np.mean(values)),
        "median_dB": float(np.median(values)),
        "std_dB": float(np.std(values, ddof=1)),
        "min_dB": float(np.min(values)),
        "q1_dB": percentile(values, 25),
        "q3_dB": percentile(values, 75),
        "max_dB": float(np.max(values)),
    }


def iqr_outliers(values):
    """
    Identify statistical outliers using the standard
    1.5 × IQR rule.
    """

    values = np.asarray(values, dtype=float)

    q1 = percentile(values, 25)
    q3 = percentile(values, 75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    indices = np.where(
        (values < lower_bound) |
        (values > upper_bound)
    )[0]

    return {
        "q1_dB": q1,
        "q3_dB": q3,
        "iqr_dB": iqr,
        "lower_bound_dB": float(lower_bound),
        "upper_bound_dB": float(upper_bound),
        "outlier_count": int(len(indices)),
        "outlier_indices": indices.tolist(),
    }


# ----------------------------------------------------------------
# 7. OVERALL DISTRIBUTIONS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL METRIC DISTRIBUTIONS")
print("=" * 70)

overall_statistics = {}
overall_outliers = {}

for metric in APPROVED_METRICS:

    values = [
        row[metric]
        for row in normalized_rows
    ]

    statistics = descriptive_statistics(values)
    outliers = iqr_outliers(values)

    overall_statistics[metric] = statistics
    overall_outliers[metric] = outliers

    print(f"\n{metric}")
    print("-" * 40)

    print(
        f"  Mean:          {statistics['mean_dB']:.3f} dB"
    )
    print(
        f"  Median:        {statistics['median_dB']:.3f} dB"
    )
    print(
        f"  Std:           {statistics['std_dB']:.3f} dB"
    )
    print(
        f"  Range:         "
        f"{statistics['min_dB']:.3f} to "
        f"{statistics['max_dB']:.3f} dB"
    )
    print(
        f"  IQR:           {outliers['iqr_dB']:.3f} dB"
    )
    print(
        f"  IQR outliers:  {outliers['outlier_count']}"
    )


# ----------------------------------------------------------------
# 8. SOURCE-BY-SOURCE DISTRIBUTIONS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE-BY-SOURCE DISTRIBUTIONS")
print("=" * 70)

source_statistics = {}
source_outliers = {}

for source in EXPECTED_SOURCES:

    source_statistics[source] = {}
    source_outliers[source] = {}

    source_rows = [
        row
        for row in normalized_rows
        if row["source"] == source
    ]

    print(f"\n{source.upper()}")
    print("-" * 70)

    for metric in APPROVED_METRICS:

        values = [
            row[metric]
            for row in source_rows
        ]

        statistics = descriptive_statistics(values)
        outliers = iqr_outliers(values)

        source_statistics[source][metric] = statistics
        source_outliers[source][metric] = outliers

        print(
            f"{metric:>7}: "
            f"mean={statistics['mean_dB']:>7.3f} | "
            f"median={statistics['median_dB']:>7.3f} | "
            f"std={statistics['std_dB']:>7.3f} | "
            f"outliers={outliers['outlier_count']:>2}"
        )


# ----------------------------------------------------------------
# 9. TRACK-LEVEL EXTREMES
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("TRACK-LEVEL EXTREME SOURCE EVALUATIONS")
print("=" * 70)

track_extremes = {}

for metric in APPROVED_METRICS:

    sorted_rows = sorted(
        normalized_rows,
        key=lambda row: row[metric]
    )

    lowest = sorted_rows[:5]
    highest = list(reversed(sorted_rows[-5:]))

    track_extremes[metric] = {
        "lowest": [
            {
                "track": row["track"],
                "source": row["source"],
                "value_dB": row[metric],
            }
            for row in lowest
        ],
        "highest": [
            {
                "track": row["track"],
                "source": row["source"],
                "value_dB": row[metric],
            }
            for row in highest
        ],
    }

    print(f"\n{metric}")

    print("  Lowest:")
    for item in track_extremes[metric]["lowest"]:
        print(
            f"    {item['value_dB']:>8.3f} dB | "
            f"{item['source']:>6} | "
            f"{item['track']}"
        )

    print("  Highest:")
    for item in track_extremes[metric]["highest"]:
        print(
            f"    {item['value_dB']:>8.3f} dB | "
            f"{item['source']:>6} | "
            f"{item['track']}"
        )


# ----------------------------------------------------------------
# 10. CREATE PLOTS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("PLOT GENERATION")
print("=" * 70)

import matplotlib.pyplot as plt


# ---------------------------------------------------------------
# Overall metric distributions
# ---------------------------------------------------------------

for metric in APPROVED_METRICS:

    values = [
        row[metric]
        for row in normalized_rows
    ]

    safe_name = metric.lower().replace("-", "_")

    plt.figure(figsize=(9, 6))

    plt.hist(
        values,
        bins=20,
        edgecolor="black",
    )

    plt.xlabel(f"{metric} (dB)")
    plt.ylabel("Number of source evaluations")
    plt.title(f"EXP002 {metric} Distribution")
    plt.grid(axis="y", alpha=0.25)

    output_path = (
        PLOTS_DIR
        / f"exp002_{safe_name}_distribution.png"
    )

    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

    print(f"✓ {output_path}")


# ---------------------------------------------------------------
# Source comparison boxplots
# ---------------------------------------------------------------

for metric in APPROVED_METRICS:

    source_values = [
        [
            row[metric]
            for row in normalized_rows
            if row["source"] == source
        ]
        for source in EXPECTED_SOURCES
    ]

    safe_name = metric.lower().replace("-", "_")

    plt.figure(figsize=(9, 6))

    plt.boxplot(
        source_values,
        tick_labels=EXPECTED_SOURCES,
        showmeans=True,
    )

    plt.xlabel("Source")
    plt.ylabel(f"{metric} (dB)")
    plt.title(
        f"EXP002 {metric} by Source"
    )
    plt.grid(axis="y", alpha=0.25)

    output_path = (
        PLOTS_DIR
        / f"exp002_{safe_name}_by_source_boxplot.png"
    )

    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

    print(f"✓ {output_path}")


# ----------------------------------------------------------------
# 11. SAVE ANALYSIS REPORT
# ----------------------------------------------------------------

analysis_report = {
    "experiment": EXPERIMENT_NAME,
    "analysis": "objective_metric_distribution_and_outlier_analysis",
    "timestamp": datetime.now().isoformat(),

    "input": {
        "csv": str(TRACK_RESULTS_CSV),
        "jsonl": str(TRACK_RESULTS_JSONL),
        "tracks": len(tracks),
        "source_evaluations": len(normalized_rows),
    },

    "approved_metrics": APPROVED_METRICS,

    "excluded_metrics": [
        "ISR",
    ],

    "overall_statistics": overall_statistics,

    "overall_iqr_outliers": overall_outliers,

    "source_statistics": source_statistics,

    "source_iqr_outliers": source_outliers,

    "track_extremes": track_extremes,

    "plots": [
        str(
            PLOTS_DIR
            / f"exp002_{metric.lower().replace('-', '_')}_distribution.png"
        )
        for metric in APPROVED_METRICS
    ] + [
        str(
            PLOTS_DIR
            / f"exp002_{metric.lower().replace('-', '_')}_by_source_boxplot.png"
        )
        for metric in APPROVED_METRICS
    ],
}

REPORT_PATH = (
    REPORTS_DIR
    / "exp002_distribution_outlier_analysis.json"
)

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        analysis_report,
        f,
        indent=2,
        allow_nan=False,
    )


# ----------------------------------------------------------------
# 12. FINAL RESULT
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 12 RESULT")
print("=" * 70)

print(f"\nTracks analysed:       {len(tracks)}")
print(f"Source evaluations:   {len(normalized_rows)}")
print(f"Approved metrics:     {len(APPROVED_METRICS)}")

print("\nAnalyses completed:")
print("  ✓ Overall distributions")
print("  ✓ Source-by-source distributions")
print("  ✓ IQR-based outlier detection")
print("  ✓ Lowest/highest source evaluations")
print("  ✓ Distribution histograms")
print("  ✓ Source comparison boxplots")

print("\nExcluded:")
print("  ✗ ISR")

print("\nAnalysis report:")
print(f"  {REPORT_PATH}")

print("\nPlots:")
for metric in APPROVED_METRICS:
    safe_name = metric.lower().replace("-", "_")
    print(
        f"  - {PLOTS_DIR / f'exp002_{safe_name}_distribution.png'}"
    )
    print(
        f"  - {PLOTS_DIR / f'exp002_{safe_name}_by_source_boxplot.png'}"
    )

print("\n" + "=" * 70)
print("✓ CELL 12 COMPLETE")
print("=" * 70)

print(
    "\nEXP002 distributions and outliers were analysed from the "
    "existing metric outputs. No WAV files were loaded and no "
    "objective metrics were recomputed."
)

EXP002 OBJECTIVE METRIC DISTRIBUTION & OUTLIER ANALYSIS

----------------------------------------------------------------------
INPUT VALIDATION
----------------------------------------------------------------------
Metrics directory: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics
Reports directory: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports
Plots directory:   G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\plots
✓ Required metric files found

----------------------------------------------------------------------
SOURCE-LEVEL DATA DISCOVERY
----------------------------------------------------------------------
CSV rows discovered: 200
✓ Correct source-level row count: 50 tracks × 4 sources

----------------------------------------------------------------------
DATA NORMALIZATION
--------

# Source-Level Comparative Analysis

In [14]:
# ================================================================
# CELL 13 — EXP002 SOURCE-LEVEL COMPARATIVE ANALYSIS
# ================================================================

import csv
import json
import math
from pathlib import Path
from datetime import datetime

print("=" * 70)
print("EXP002 SOURCE-LEVEL COMPARATIVE ANALYSIS")
print("=" * 70)

# ----------------------------------------------------------------
# 1. CONFIGURATION DISCOVERY
# ----------------------------------------------------------------

if "EXPERIMENT_NAME" not in globals():
    EXPERIMENT_NAME = "EXP002"

if "METRICS_DIR" not in globals():
    raise RuntimeError(
        "METRICS_DIR is not defined. "
        "Run the earlier configuration cells before Cell 13."
    )

if "REPORTS_DIR" not in globals():
    raise RuntimeError(
        "REPORTS_DIR is not defined. "
        "Run the earlier configuration cells before Cell 13."
    )

METRICS_DIR = Path(METRICS_DIR)
REPORTS_DIR = Path(REPORTS_DIR)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = METRICS_DIR / "per_track_metrics.csv"
JSONL_PATH = METRICS_DIR / "per_track_metrics.jsonl"

OUTPUT_REPORT = (
    REPORTS_DIR /
    "exp002_source_level_comparative_analysis.json"
)

EXPECTED_TRACKS = 50

EXPECTED_SOURCES = [
    "vocals",
    "drums",
    "bass",
    "other",
]

APPROVED_METRICS = [
    "SDR",
    "SIR",
    "SAR",
    "SI-SDR",
]

CSV_COLUMNS = {
    "SDR": "SDR_dB",
    "SIR": "SIR_dB",
    "SAR": "SAR_dB",
    "SI-SDR": "SI_SDR_dB",
}


# ----------------------------------------------------------------
# 2. INPUT VALIDATION
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("INPUT VALIDATION")
print("-" * 70)

print(f"Metrics directory: {METRICS_DIR}")
print(f"Reports directory: {REPORTS_DIR}")

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Required CSV not found: {CSV_PATH}"
    )

if not JSONL_PATH.exists():
    raise FileNotFoundError(
        f"Required JSONL not found: {JSONL_PATH}"
    )

print(f"✓ CSV found:   {CSV_PATH}")
print(f"✓ JSONL found:  {JSONL_PATH}")


# ----------------------------------------------------------------
# 3. LOAD SOURCE-LEVEL CSV DATA
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("SOURCE-LEVEL DATA LOADING")
print("-" * 70)

rows = []

with open(
    CSV_PATH,
    "r",
    encoding="utf-8",
    newline="",
) as f:

    reader = csv.DictReader(f)

    required_columns = [
        "track",
        "source",
        *CSV_COLUMNS.values(),
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in reader.fieldnames
    ]

    if missing_columns:
        raise ValueError(
            "CSV is missing required columns:\n"
            + "\n".join(
                f"  - {column}"
                for column in missing_columns
            )
        )

    for row in reader:

        track = row["track"].strip()
        source = row["source"].strip().lower()

        if not track:
            raise ValueError("Encountered CSV row with empty track.")

        if source not in EXPECTED_SOURCES:
            raise ValueError(
                f"Unexpected source '{source}' "
                f"found in CSV."
            )

        normalized = {
            "track": track,
            "source": source,
        }

        for metric, column in CSV_COLUMNS.items():

            try:
                value = float(row[column])
            except (TypeError, ValueError):
                raise ValueError(
                    f"Non-numeric {metric} value for "
                    f"{track} / {source}: {row[column]}"
                )

            if not math.isfinite(value):
                raise ValueError(
                    f"Non-finite {metric} value for "
                    f"{track} / {source}: {value}"
                )

            normalized[metric] = value

        rows.append(normalized)


expected_rows = EXPECTED_TRACKS * len(EXPECTED_SOURCES)

print(f"CSV rows discovered: {len(rows)}")

if len(rows) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows} source-level rows "
        f"but found {len(rows)}."
    )

print(
    f"✓ Correct source-level row count: "
    f"{EXPECTED_TRACKS} tracks × "
    f"{len(EXPECTED_SOURCES)} sources = {expected_rows}"
)


# ----------------------------------------------------------------
# 4. TRACK / SOURCE COVERAGE
# ----------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK / SOURCE COVERAGE")
print("-" * 70)

tracks = sorted(
    {row["track"] for row in rows},
    key=str.lower,
)

if len(tracks) != EXPECTED_TRACKS:
    raise ValueError(
        f"Expected {EXPECTED_TRACKS} unique tracks, "
        f"found {len(tracks)}."
    )

print(f"Unique tracks: {len(tracks)}")

for source in EXPECTED_SOURCES:

    source_tracks = {
        row["track"]
        for row in rows
        if row["source"] == source
    }

    print(
        f"{source:>7}: "
        f"{len(source_tracks)}/{EXPECTED_TRACKS}"
    )

    if len(source_tracks) != EXPECTED_TRACKS:
        raise ValueError(
            f"Incomplete coverage for source '{source}'."
        )

print("✓ Complete source coverage confirmed")


# ----------------------------------------------------------------
# 5. SOURCE-LEVEL GROUPING
# ----------------------------------------------------------------

source_values = {
    source: {
        metric: []
        for metric in APPROVED_METRICS
    }
    for source in EXPECTED_SOURCES
}

for row in rows:

    source = row["source"]

    for metric in APPROVED_METRICS:
        source_values[source][metric].append(
            row[metric]
        )


# ----------------------------------------------------------------
# 6. SOURCE MEAN CALCULATION
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE MEAN COMPARISON")
print("=" * 70)

source_means = {
    source: {}
    for source in EXPECTED_SOURCES
}

for source in EXPECTED_SOURCES:

    for metric in APPROVED_METRICS:

        values = source_values[source][metric]

        source_means[source][metric] = (
            sum(values) / len(values)
        )

print()

header = (
    f"{'Source':<10}"
    + "".join(
        f"{metric:>12}"
        for metric in APPROVED_METRICS
    )
)

print(header)
print("-" * len(header))

for source in EXPECTED_SOURCES:

    line = f"{source:<10}"

    for metric in APPROVED_METRICS:

        line += (
            f"{source_means[source][metric]:>12.3f}"
        )

    print(line)


# ----------------------------------------------------------------
# 7. SOURCE RANKING BY METRIC
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE RANKING BY METRIC")
print("=" * 70)

source_rankings = {}

for metric in APPROVED_METRICS:

    ranked = sorted(
        EXPECTED_SOURCES,
        key=lambda source: source_means[source][metric],
        reverse=True,
    )

    source_rankings[metric] = ranked

    print(f"\n{metric}")
    print("-" * 40)

    for rank, source in enumerate(ranked, start=1):

        value = source_means[source][metric]

        print(
            f"  {rank}. {source:<7} "
            f"{value:.3f} dB"
        )


# ----------------------------------------------------------------
# 8. BEST / WORST SOURCE AND PERFORMANCE GAP
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST / WORST SOURCE ANALYSIS")
print("=" * 70)

metric_comparison = {}

for metric in APPROVED_METRICS:

    ranked = source_rankings[metric]

    best_source = ranked[0]
    worst_source = ranked[-1]

    best_value = source_means[best_source][metric]
    worst_value = source_means[worst_source][metric]

    gap = best_value - worst_value

    metric_comparison[metric] = {
        "best_source": best_source,
        "best_mean_dB": best_value,
        "worst_source": worst_source,
        "worst_mean_dB": worst_value,
        "best_worst_gap_dB": gap,
    }

    print(f"\n{metric}")
    print(
        f"  Best:  {best_source} "
        f"({best_value:.3f} dB)"
    )
    print(
        f"  Worst: {worst_source} "
        f"({worst_value:.3f} dB)"
    )
    print(
        f"  Gap:   {gap:.3f} dB"
    )


# ----------------------------------------------------------------
# 9. SOURCE PROFILE
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE PERFORMANCE PROFILE")
print("=" * 70)

source_profiles = {}

for source in EXPECTED_SOURCES:

    ranking_positions = {}

    for metric in APPROVED_METRICS:

        ranking_positions[metric] = (
            source_rankings[metric].index(source) + 1
        )

    average_rank = (
        sum(ranking_positions.values())
        / len(ranking_positions)
    )

    source_profiles[source] = {
        "means_dB": source_means[source],
        "rankings": ranking_positions,
        "average_rank": average_rank,
    }

    print(f"\n{source.upper()}")

    for metric in APPROVED_METRICS:

        print(
            f"  {metric:<6}: "
            f"{source_means[source][metric]:.3f} dB "
            f"(rank {ranking_positions[metric]}/"
            f"{len(EXPECTED_SOURCES)})"
        )

    print(
        f"  Average rank: {average_rank:.2f}"
    )


# ----------------------------------------------------------------
# 10. OVERALL SOURCE RANKING
# ----------------------------------------------------------------

overall_source_ranking = sorted(
    EXPECTED_SOURCES,
    key=lambda source: source_profiles[source]["average_rank"],
)

print("\n" + "=" * 70)
print("OVERALL SOURCE RANKING")
print("=" * 70)

for rank, source in enumerate(
    overall_source_ranking,
    start=1,
):

    print(
        f"  {rank}. {source:<7} "
        f"(average rank: "
        f"{source_profiles[source]['average_rank']:.2f})"
    )


# ----------------------------------------------------------------
# 11. TRACK-LEVEL BEST / WORST SOURCE ANALYSIS
# ----------------------------------------------------------------
#
# This does NOT recompute any metric.
# It simply compares the already calculated values within each track.
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("TRACK-LEVEL SOURCE COMPARISON")
print("=" * 70)

track_source_values = {}

for row in rows:

    track = row["track"]
    source = row["source"]

    track_source_values.setdefault(
        track,
        {}
    )[source] = {
        metric: row[metric]
        for metric in APPROVED_METRICS
    }


track_level_summary = {}

for metric in APPROVED_METRICS:

    best_counts = {
        source: 0
        for source in EXPECTED_SOURCES
    }

    worst_counts = {
        source: 0
        for source in EXPECTED_SOURCES
    }

    for track in tracks:

        values = {
            source: track_source_values[track][source][metric]
            for source in EXPECTED_SOURCES
        }

        best_source = max(
            values,
            key=values.get,
        )

        worst_source = min(
            values,
            key=values.get,
        )

        best_counts[best_source] += 1
        worst_counts[worst_source] += 1

    track_level_summary[metric] = {
        "best_source_count": best_counts,
        "worst_source_count": worst_counts,
    }

    print(f"\n{metric}")

    print("  Tracks where each source is best:")

    for source in EXPECTED_SOURCES:

        print(
            f"    {source:<7}: "
            f"{best_counts[source]}/{EXPECTED_TRACKS}"
        )

    print("  Tracks where each source is worst:")

    for source in EXPECTED_SOURCES:

        print(
            f"    {source:<7}: "
            f"{worst_counts[source]}/{EXPECTED_TRACKS}"
        )


# ----------------------------------------------------------------
# 12. CONSOLIDATED REPORT
# ----------------------------------------------------------------

report = {
    "experiment": EXPERIMENT_NAME,
    "generated_at": datetime.now().isoformat(),

    "scope": {
        "tracks": EXPECTED_TRACKS,
        "sources": EXPECTED_SOURCES,
        "approved_metrics": APPROVED_METRICS,
        "excluded_metrics": [
            "ISR"
        ],
    },

    "input": {
        "csv": str(CSV_PATH),
        "jsonl": str(JSONL_PATH),
        "source_level_rows": len(rows),
    },

    "source_means_dB": source_means,

    "source_rankings": source_rankings,

    "best_worst_comparison": metric_comparison,

    "source_profiles": source_profiles,

    "overall_source_ranking": overall_source_ranking,

    "track_level_source_comparison": track_level_summary,

    "policy": {
        "wav_files_loaded": False,
        "metrics_recomputed": False,
        "isr_used_downstream": False,
    },
}


with open(
    OUTPUT_REPORT,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        report,
        f,
        indent=2,
    )


# ----------------------------------------------------------------
# 13. FINAL RESULT
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 13 RESULT")
print("=" * 70)

print(f"\nTracks analysed:       {EXPECTED_TRACKS}")
print(
    f"Source evaluations:   {len(rows)}"
)
print(
    f"Approved metrics:     {len(APPROVED_METRICS)}"
)

print("\nApproved metrics:")
for metric in APPROVED_METRICS:
    print(f"  ✓ {metric}")

print("\nExcluded:")
print("  ✗ ISR")

print("\nAnalyses completed:")
print("  ✓ Source mean comparison")
print("  ✓ Source ranking by metric")
print("  ✓ Best/worst source identification")
print("  ✓ Best/worst performance gaps")
print("  ✓ Source performance profiles")
print("  ✓ Overall source ranking")
print("  ✓ Track-level source comparison")

print("\nReport:")
print(f"  {OUTPUT_REPORT}")

print("\n" + "=" * 70)
print("✓ CELL 13 COMPLETE")
print("=" * 70)

print(
    "\nEXP002 source-level comparative analysis was completed "
    "from the existing metric outputs."
)
print(
    "No WAV files were loaded and no objective metrics were recomputed."
)

print(
    "\n" + datetime.now().isoformat()
)

EXP002 SOURCE-LEVEL COMPARATIVE ANALYSIS

----------------------------------------------------------------------
INPUT VALIDATION
----------------------------------------------------------------------
Metrics directory: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics
Reports directory: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports
✓ CSV found:   G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics\per_track_metrics.csv
✓ JSONL found:  G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics\per_track_metrics.jsonl

----------------------------------------------------------------------
SOURCE-LEVEL DATA LOADING
----------------------------------------------------------------------
CSV rows discovered: 200
✓ Correct source-level row

# Track-Level Aggregate Analysis

In [17]:
# ======================================================================
# EXP002 TRACK-LEVEL AGGREGATE ANALYSIS
# ======================================================================

import json
import os
import math
import statistics
from collections import defaultdict

print("=" * 70)
print("EXP002 TRACK-LEVEL AGGREGATE ANALYSIS")
print("=" * 70)

# ----------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------

EXP2_ROOT = r"G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002"

METRICS_DIR = os.path.join(EXP2_ROOT, "metrics")
REPORTS_DIR = os.path.join(EXP2_ROOT, "reports")

JSONL_PATH = os.path.join(
    METRICS_DIR,
    "per_track_metrics.jsonl"
)

REPORT_PATH = os.path.join(
    REPORTS_DIR,
    "exp002_track_level_aggregate_analysis.json"
)

EXPECTED_SOURCES = [
    "vocals",
    "drums",
    "bass",
    "other"
]

APPROVED_METRICS = [
    "SDR",
    "SIR",
    "SAR",
    "SI-SDR"
]

# Actual JSONL metric keys
METRIC_KEYS = {
    "SDR": "SDR_dB",
    "SIR": "SIR_dB",
    "SAR": "SAR_dB",
    "SI-SDR": "SI_SDR_dB"
}

# ----------------------------------------------------------------------
# INPUT VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("INPUT VALIDATION")
print("-" * 70)

print(f"Metrics directory: {METRICS_DIR}")
print(f"Reports directory: {REPORTS_DIR}")

if not os.path.isfile(JSONL_PATH):
    raise FileNotFoundError(
        f"Required JSONL file not found:\n{JSONL_PATH}"
    )

print(f"✓ JSONL found: {JSONL_PATH}")

# ----------------------------------------------------------------------
# LOAD TRACK-LEVEL JSONL
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK-LEVEL DATA LOADING")
print("-" * 70)

records = []

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Invalid JSON on line {line_number}: {exc}"
            )

        records.append(record)

print(f"JSONL records discovered: {len(records)}")

if len(records) != 50:
    raise ValueError(
        f"Expected 50 track records, found {len(records)}"
    )

print("✓ Correct track-level record count")

# ----------------------------------------------------------------------
# VALIDATE TRACK RECORDS
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK-LEVEL RECORD VALIDATION")
print("-" * 70)

track_ids = []
track_average_values = {}

for record_index, record in enumerate(records, start=1):

    if record.get("status") != "success":
        raise ValueError(
            f"Record {record_index} does not have status='success'"
        )

    track = record.get("track")

    if not track:
        raise ValueError(
            f"Record {record_index} has no track identifier"
        )

    track_ids.append(track)

    # --------------------------------------------------------------
    # Validate source-level structures
    # --------------------------------------------------------------

    sources = record.get("sources")

    if not isinstance(sources, dict):
        raise ValueError(
            f"Missing or invalid 'sources' structure for {track}"
        )

    for source in EXPECTED_SOURCES:

        if source not in sources:
            raise ValueError(
                f"Missing source '{source}' for {track}"
            )

        source_data = sources[source]

        if not isinstance(source_data, dict):
            raise ValueError(
                f"Invalid source data for {track} / {source}"
            )

        for metric in APPROVED_METRICS:

            key = METRIC_KEYS[metric]

            if key not in source_data:
                raise ValueError(
                    f"Missing {key} for {track} / {source}"
                )

            value = source_data[key]

            if not isinstance(value, (int, float)) or not math.isfinite(value):
                raise ValueError(
                    f"Invalid {metric} value for {track} / {source}: {value}"
                )

    # --------------------------------------------------------------
    # Validate stored track averages
    # --------------------------------------------------------------

    track_average = record.get("track_average")

    if not isinstance(track_average, dict):
        raise ValueError(
            f"Missing or invalid 'track_average' for {track}"
        )

    track_average_values[track] = {}

    for metric in APPROVED_METRICS:

        key = METRIC_KEYS[metric]

        if key not in track_average:
            raise ValueError(
                f"Missing track-average {key} for {track}"
            )

        value = track_average[key]

        if not isinstance(value, (int, float)) or not math.isfinite(value):
            raise ValueError(
                f"Invalid track-average {metric} for {track}: {value}"
            )

        track_average_values[track][metric] = float(value)

print(f"Tracks validated: {len(track_ids)}")
print("✓ All source-level metric structures are valid")
print("✓ All stored track-average metrics are valid")

# ----------------------------------------------------------------------
# UNIQUE TRACK VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK IDENTIFIER VALIDATION")
print("-" * 70)

unique_tracks = set(track_ids)

print(f"Track identifiers discovered: {len(track_ids)}")
print(f"Unique tracks discovered:     {len(unique_tracks)}")

if len(unique_tracks) != len(track_ids):
    duplicates = [
        track for track in set(track_ids)
        if track_ids.count(track) > 1
    ]

    raise ValueError(
        f"Duplicate track identifiers detected: {duplicates}"
    )

print("✓ All 50 track identifiers are unique")

# ----------------------------------------------------------------------
# TRACK-AVERAGE DESCRIPTIVE STATISTICS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TRACK-AVERAGE METRIC STATISTICS")
print("=" * 70)

aggregate_statistics = {}

for metric in APPROVED_METRICS:

    values = [
        track_average_values[track][metric]
        for track in track_ids
    ]

    sorted_values = sorted(values)

    count = len(values)
    mean_value = statistics.mean(values)
    median_value = statistics.median(values)

    if count > 1:
        std_value = statistics.stdev(values)
    else:
        std_value = 0.0

    minimum = min(values)
    maximum = max(values)

    # Percentiles using linear interpolation
    def percentile(data, p):
        if len(data) == 1:
            return data[0]

        position = (len(data) - 1) * p
        lower = int(math.floor(position))
        upper = int(math.ceil(position))

        if lower == upper:
            return data[lower]

        fraction = position - lower

        return (
            data[lower]
            + fraction * (data[upper] - data[lower])
        )

    q1 = percentile(sorted_values, 0.25)
    q3 = percentile(sorted_values, 0.75)

    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    outlier_count = sum(
        value < lower_fence or value > upper_fence
        for value in values
    )

    aggregate_statistics[metric] = {
        "count": count,
        "mean_dB": mean_value,
        "median_dB": median_value,
        "std_dB": std_value,
        "min_dB": minimum,
        "q1_dB": q1,
        "q3_dB": q3,
        "max_dB": maximum,
        "iqr_dB": iqr,
        "lower_fence_dB": lower_fence,
        "upper_fence_dB": upper_fence,
        "iqr_outlier_count": outlier_count
    }

    print(f"\n{metric}")
    print("-" * 40)
    print(f"  Count:        {count}")
    print(f"  Mean:         {mean_value:.3f} dB")
    print(f"  Median:       {median_value:.3f} dB")
    print(f"  Std:          {std_value:.3f} dB")
    print(f"  Min:          {minimum:.3f} dB")
    print(f"  Q1:           {q1:.3f} dB")
    print(f"  Q3:           {q3:.3f} dB")
    print(f"  Max:          {maximum:.3f} dB")
    print(f"  IQR:          {iqr:.3f} dB")
    print(f"  IQR outliers: {outlier_count}")

# ----------------------------------------------------------------------
# TRACK-LEVEL RANKING
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TRACK-LEVEL PERFORMANCE RANKING")
print("=" * 70)

track_rankings = {}

for metric in APPROVED_METRICS:

    ranking = sorted(
        [
            {
                "track": track,
                "value_dB": track_average_values[track][metric]
            }
            for track in track_ids
        ],
        key=lambda x: x["value_dB"],
        reverse=True
    )

    track_rankings[metric] = ranking

    print(f"\n{metric}")
    print("-" * 40)

    print("  Highest:")
    for item in ranking[:5]:
        print(
            f"    {item['value_dB']:8.3f} dB | "
            f"{item['track']}"
        )

    print("  Lowest:")
    for item in ranking[-5:][::-1]:
        print(
            f"    {item['value_dB']:8.3f} dB | "
            f"{item['track']}"
        )

# ----------------------------------------------------------------------
# BEST / WORST TRACK AGGREGATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TRACK PERFORMANCE SUMMARY")
print("=" * 70)

track_summary = {}

for track in track_ids:

    values = {
        metric: track_average_values[track][metric]
        for metric in APPROVED_METRICS
    }

    mean_across_metrics = statistics.mean(values.values())

    best_metric = max(
        values,
        key=values.get
    )

    worst_metric = min(
        values,
        key=values.get
    )

    track_summary[track] = {
        "metrics": values,
        "mean_across_approved_metrics_dB": mean_across_metrics,
        "best_metric": best_metric,
        "worst_metric": worst_metric
    }

# Rank tracks by their mean across the four approved metrics
overall_track_ranking = sorted(
    [
        {
            "track": track,
            "mean_across_approved_metrics_dB":
                track_summary[track]["mean_across_approved_metrics_dB"]
        }
        for track in track_ids
    ],
    key=lambda x: x["mean_across_approved_metrics_dB"],
    reverse=True
)

print("\nOverall track ranking using mean of approved metrics:")

print("\n  Highest-performing tracks:")
for item in overall_track_ranking[:5]:
    print(
        f"    {item['mean_across_approved_metrics_dB']:.3f} dB | "
        f"{item['track']}"
    )

print("\n  Lowest-performing tracks:")
for item in overall_track_ranking[-5:][::-1]:
    print(
        f"    {item['mean_across_approved_metrics_dB']:.3f} dB | "
        f"{item['track']}"
    )

# ----------------------------------------------------------------------
# CROSS-CHECK STORED TRACK AVERAGES
# AGAINST SOURCE-LEVEL VALUES
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TRACK-AVERAGE CONSISTENCY CHECK")
print("=" * 70)

consistency_tolerance = 1e-9
consistency_checks = []

for record in records:

    track = record["track"]
    sources = record["sources"]
    stored_average = record["track_average"]

    for metric in APPROVED_METRICS:

        key = METRIC_KEYS[metric]

        source_values = [
            float(sources[source][key])
            for source in EXPECTED_SOURCES
        ]

        calculated_average = statistics.mean(source_values)
        stored_value = float(stored_average[key])

        difference = abs(
            calculated_average - stored_value
        )

        consistent = difference <= consistency_tolerance

        consistency_checks.append({
            "track": track,
            "metric": metric,
            "stored_track_average_dB": stored_value,
            "calculated_source_mean_dB": calculated_average,
            "absolute_difference_dB": difference,
            "consistent": consistent
        })

        if not consistent:
            raise ValueError(
                f"Track-average inconsistency detected for "
                f"{track} / {metric}: "
                f"stored={stored_value}, "
                f"calculated={calculated_average}"
            )

print(
    f"Consistency checks performed: "
    f"{len(consistency_checks)}"
)

print("✓ All stored track averages equal the mean of the four source values")

# ----------------------------------------------------------------------
# SAVE REPORT
# ----------------------------------------------------------------------

report = {
    "experiment": "EXP002",
    "analysis": "track_level_aggregate_analysis",

    "data_source": {
        "jsonl": JSONL_PATH,
        "tracks": len(track_ids),
        "sources_per_track": len(EXPECTED_SOURCES),
        "source_evaluations": len(track_ids) * len(EXPECTED_SOURCES)
    },

    "approved_metrics": APPROVED_METRICS,

    "excluded_metrics": [
        "ISR"
    ],

    "track_average_statistics": aggregate_statistics,

    "track_rankings": track_rankings,

    "track_summary": track_summary,

    "overall_track_ranking": overall_track_ranking,

    "track_average_consistency": {
        "checks_performed": len(consistency_checks),
        "all_consistent": all(
            item["consistent"]
            for item in consistency_checks
        ),
        "tolerance_dB": consistency_tolerance
    }
}

os.makedirs(REPORTS_DIR, exist_ok=True)

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )

# ----------------------------------------------------------------------
# FINAL RESULT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 14 RESULT")
print("=" * 70)

print(f"\nTracks analysed:       {len(track_ids)}")
print(
    f"Source evaluations:   "
    f"{len(track_ids) * len(EXPECTED_SOURCES)}"
)
print(f"Approved metrics:     {len(APPROVED_METRICS)}")

print("\nApproved metrics:")
for metric in APPROVED_METRICS:
    print(f"  ✓ {metric}")

print("\nExcluded:")
print("  ✗ ISR")

print("\nAnalyses completed:")
print("  ✓ Track-average descriptive statistics")
print("  ✓ Track-level metric rankings")
print("  ✓ Highest/lowest performing tracks")
print("  ✓ Cross-metric track performance summary")
print("  ✓ Stored track-average consistency validation")

print("\nReport:")
print(f"  {REPORT_PATH}")

print("\n" + "=" * 70)
print("✓ CELL 14 COMPLETE")
print("=" * 70)

print(
    "\nEXP002 track-level aggregate analysis was completed "
    "from the existing metric outputs."
)

print(
    "No WAV files were loaded and no objective metrics were recomputed."
)

EXP002 TRACK-LEVEL AGGREGATE ANALYSIS

----------------------------------------------------------------------
INPUT VALIDATION
----------------------------------------------------------------------
Metrics directory: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics
Reports directory: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports
✓ JSONL found: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics\per_track_metrics.jsonl

----------------------------------------------------------------------
TRACK-LEVEL DATA LOADING
----------------------------------------------------------------------
JSONL records discovered: 50
✓ Correct track-level record count

----------------------------------------------------------------------
TRACK-LEVEL RECORD VALIDATION
-------------------------------------------

# Consolidated Objective Analysis Summary

In [20]:
# ======================================================================
# EXP002 CONSOLIDATED OBJECTIVE ANALYSIS SUMMARY
# ======================================================================

import json
from pathlib import Path
from datetime import datetime


print("=" * 70)
print("EXP002 CONSOLIDATED OBJECTIVE ANALYSIS SUMMARY")
print("=" * 70)


# ----------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------

EXP_NAME = "EXP002"

BASE_DIR = Path(
    r"G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation"
)

EXP_DIR = (
    BASE_DIR
    / "05_Experiments"
    / "Evaluation"
    / "Objective"
    / EXP_NAME
)

METRICS_DIR = EXP_DIR / "metrics"
REPORTS_DIR = EXP_DIR / "reports"


# ----------------------------------------------------------------------
# REPORT PATHS
# ----------------------------------------------------------------------

REPORT_PATHS = {
    "integrity": REPORTS_DIR / "metrics_integrity_audit.json",
    "descriptive": REPORTS_DIR / "exp002_metric_descriptive_statistics.json",
    "distribution": REPORTS_DIR / "exp002_distribution_outlier_analysis.json",
    "source_comparison": REPORTS_DIR / "exp002_source_level_comparative_analysis.json",
    "track_aggregate": REPORTS_DIR / "exp002_track_level_aggregate_analysis.json",
}


# ----------------------------------------------------------------------
# APPROVED METRICS
# ----------------------------------------------------------------------

APPROVED_METRICS = [
    "SDR",
    "SIR",
    "SAR",
    "SI-SDR",
]

EXCLUDED_METRICS = [
    "ISR",
]


# ----------------------------------------------------------------------
# INPUT VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("INPUT VALIDATION")
print("-" * 70)

for name, path in REPORT_PATHS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Required report not found: {path}"
        )

    print(f"✓ {name.replace('_', ' ').title()}: {path}")


# ----------------------------------------------------------------------
# LOAD EXISTING REPORTS
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("LOADING EXISTING EXP002 REPORTS")
print("-" * 70)

loaded_reports = {}

for name, path in REPORT_PATHS.items():

    with open(path, "r", encoding="utf-8") as f:
        loaded_reports[name] = json.load(f)

    print(f"✓ {name.replace('_', ' ').title()} loaded")


integrity = loaded_reports["integrity"]
descriptive = loaded_reports["descriptive"]
distribution = loaded_reports["distribution"]
source_comparison = loaded_reports["source_comparison"]
track_aggregate = loaded_reports["track_aggregate"]


# ----------------------------------------------------------------------
# CORE DATASET COUNTS
# ----------------------------------------------------------------------

dataset_info = descriptive["dataset"]

tracks_analyzed = dataset_info["tracks"]
sources_per_track = dataset_info["sources_per_track"]
source_evaluations = dataset_info["source_evaluations"]


print("\n" + "-" * 70)
print("CORE EXP002 COUNTS")
print("-" * 70)

print(f"Tracks analysed:       {tracks_analyzed}")
print(f"Sources per track:     {sources_per_track}")
print(f"Source evaluations:    {source_evaluations}")
print(f"Approved metrics:      {len(APPROVED_METRICS)}")
print(f"Excluded metric:       ISR")


# ----------------------------------------------------------------------
# CONSOLIDATED METRIC STATISTICS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CONSOLIDATED METRIC STATISTICS")
print("=" * 70)

# IMPORTANT:
# The descriptive report stores overall source-level statistics under:
# "overall_source_level_statistics"

overall_source_statistics = {}

for metric in APPROVED_METRICS:

    stats = descriptive["overall_source_level_statistics"][metric]

    overall_source_statistics[metric] = stats

    print(f"\n{metric}")
    print("-" * 40)
    print(f"  Count:   {stats['count']}")
    print(f"  Mean:    {stats['mean_dB']:.3f} dB")
    print(f"  Median:  {stats['median_dB']:.3f} dB")
    print(f"  Std:     {stats['std_dB']:.3f} dB")
    print(f"  Min:     {stats['min_dB']:.3f} dB")
    print(f"  Q1:      {stats['q1_dB']:.3f} dB")
    print(f"  Q3:      {stats['q3_dB']:.3f} dB")
    print(f"  Max:     {stats['max_dB']:.3f} dB")
    print(f"  IQR:     {stats['iqr_dB']:.3f} dB")


# ----------------------------------------------------------------------
# TRACK-AVERAGE STATISTICS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TRACK-AVERAGE STATISTICS")
print("=" * 70)

track_average_statistics = {}

for metric in APPROVED_METRICS:

    stats = descriptive["track_average_statistics"][metric]

    track_average_statistics[metric] = stats

    print(f"\n{metric}")
    print("-" * 40)
    print(f"  Count:   {stats['count']}")
    print(f"  Mean:    {stats['mean_dB']:.3f} dB")
    print(f"  Median:  {stats['median_dB']:.3f} dB")
    print(f"  Std:     {stats['std_dB']:.3f} dB")
    print(f"  Min:     {stats['min_dB']:.3f} dB")
    print(f"  Q1:      {stats['q1_dB']:.3f} dB")
    print(f"  Q3:      {stats['q3_dB']:.3f} dB")
    print(f"  Max:     {stats['max_dB']:.3f} dB")
    print(f"  IQR:     {stats['iqr_dB']:.3f} dB")


# ----------------------------------------------------------------------
# SOURCE PERFORMANCE SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE PERFORMANCE SUMMARY")
print("=" * 70)

source_level_statistics = descriptive["source_level_statistics"]

source_summary = {}

for source in source_level_statistics:

    source_summary[source] = {}

    print(f"\n{source.upper()}")
    print("-" * 40)

    for metric in APPROVED_METRICS:

        stats = source_level_statistics[source][metric]

        source_summary[source][metric] = {
            "mean_dB": stats["mean_dB"],
            "median_dB": stats["median_dB"],
            "std_dB": stats["std_dB"],
            "min_dB": stats["min_dB"],
            "max_dB": stats["max_dB"],
        }

        print(
            f"  {metric:<6}: "
            f"{stats['mean_dB']:.3f} dB"
        )


# ----------------------------------------------------------------------
# SOURCE RANKING SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE RANKING SUMMARY")
print("=" * 70)

source_rankings = {}

for metric in APPROVED_METRICS:

    ranking = sorted(
        source_level_statistics.keys(),
        key=lambda source: source_level_statistics[source][metric]["mean_dB"],
        reverse=True,
    )

    source_rankings[metric] = ranking

    print(f"\n{metric}")
    print("-" * 40)

    for index, source in enumerate(ranking, start=1):

        value = source_level_statistics[source][metric]["mean_dB"]

        print(
            f"  {index}. {source:<7} {value:.3f} dB"
        )


# ----------------------------------------------------------------------
# BEST / WORST SOURCE
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST / WORST SOURCE SUMMARY")
print("=" * 70)

best_worst_sources = {}

for metric in APPROVED_METRICS:

    ranking = source_rankings[metric]

    best_source = ranking[0]
    worst_source = ranking[-1]

    best_value = source_level_statistics[best_source][metric]["mean_dB"]
    worst_value = source_level_statistics[worst_source][metric]["mean_dB"]

    gap = best_value - worst_value

    best_worst_sources[metric] = {
        "best_source": best_source,
        "best_mean_dB": best_value,
        "worst_source": worst_source,
        "worst_mean_dB": worst_value,
        "gap_dB": gap,
    }

    print(f"\n{metric}")
    print(f"  Best :  {best_source} ({best_value:.3f} dB)")
    print(f"  Worst:  {worst_source} ({worst_value:.3f} dB)")
    print(f"  Gap  :  {gap:.3f} dB")


# ----------------------------------------------------------------------
# OUTLIER SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("OUTLIER SUMMARY")
print("=" * 70)

outlier_summary = {}

distribution_overall = distribution.get(
    "overall_distributions",
    distribution.get("overall", {})
)

for metric in APPROVED_METRICS:

    metric_data = distribution_overall.get(metric, {})

    outlier_count = metric_data.get(
        "iqr_outliers",
        metric_data.get("outliers", None)
    )

    outlier_summary[metric] = outlier_count

    if outlier_count is not None:
        print(f"  {metric:<6}: {outlier_count} IQR outliers")
    else:
        print(f"  {metric:<6}: outlier count not available")


# ----------------------------------------------------------------------
# TRACK PERFORMANCE SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TRACK PERFORMANCE SUMMARY")
print("=" * 70)

track_summary = {}

# Cell 14 already calculated the cross-metric track summary.
# Preserve it without recalculating any metrics.

if isinstance(track_aggregate, dict):

    if "track_performance_summary" in track_aggregate:
        track_summary = track_aggregate["track_performance_summary"]

    elif "performance_summary" in track_aggregate:
        track_summary = track_aggregate["performance_summary"]

    else:
        track_summary = {}


if track_summary:
    print("✓ Existing track-level performance summary incorporated")
else:
    print("✓ Track-level aggregate report loaded")


# ----------------------------------------------------------------------
# CONSOLIDATED REPORT
# ----------------------------------------------------------------------

consolidated_report = {
    "experiment": EXP_NAME,
    "generated_at": datetime.now().isoformat(),

    "dataset": {
        "tracks": tracks_analyzed,
        "sources_per_track": sources_per_track,
        "source_evaluations": source_evaluations,
    },

    "approved_metrics": APPROVED_METRICS,
    "excluded_metrics": EXCLUDED_METRICS,

    "metric_statistics": {
        "overall_source_level": overall_source_statistics,
        "track_average": track_average_statistics,
    },

    "source_performance": source_summary,

    "source_rankings": source_rankings,

    "best_worst_sources": best_worst_sources,

    "outlier_summary": outlier_summary,

    "track_performance_summary": track_summary,

    "input_reports": {
        key: str(path)
        for key, path in REPORT_PATHS.items()
    },

    "integrity_policy": {
        "wav_files_loaded": False,
        "metrics_recomputed": False,
        "existing_evaluation_outputs_used": True,
        "isr_included_downstream": False,
    },
}


# ----------------------------------------------------------------------
# SAVE CONSOLIDATED REPORT
# ----------------------------------------------------------------------

OUTPUT_PATH = REPORTS_DIR / "exp002_consolidated_objective_analysis.json"

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        consolidated_report,
        f,
        indent=2,
    )


# ----------------------------------------------------------------------
# FINAL RESULT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 15 RESULT")
print("=" * 70)

print(f"\nTracks analysed:       {tracks_analyzed}")
print(f"Source evaluations:   {source_evaluations}")
print(f"Approved metrics:     {len(APPROVED_METRICS)}")

print("\nApproved metrics:")
for metric in APPROVED_METRICS:
    print(f"  ✓ {metric}")

print("\nExcluded:")
print("  ✗ ISR")

print("\nConsolidated analyses:")
print("  ✓ Overall source-level statistics")
print("  ✓ Track-average statistics")
print("  ✓ Source performance summary")
print("  ✓ Source rankings")
print("  ✓ Best/worst source analysis")
print("  ✓ Outlier summary")
print("  ✓ Existing track-level aggregate results")

print("\nConsolidated report:")
print(f"  {OUTPUT_PATH}")

print("\n" + "=" * 70)
print("✓ CELL 15 COMPLETE")
print("=" * 70)

print(
    "\nEXP002 consolidated objective analysis was generated "
    "from the existing evaluation reports."
)

print(
    "No WAV files were loaded and no objective metrics were recomputed."
)

EXP002 CONSOLIDATED OBJECTIVE ANALYSIS SUMMARY

----------------------------------------------------------------------
INPUT VALIDATION
----------------------------------------------------------------------
✓ Integrity: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports\metrics_integrity_audit.json
✓ Descriptive: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports\exp002_metric_descriptive_statistics.json
✓ Distribution: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports\exp002_distribution_outlier_analysis.json
✓ Source Comparison: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports\exp002_source_level_comparative_analysis.json
✓ Track Aggregate: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evalua

# Final Report Consistency & Quality-Control Audit

In [21]:
# ============================================================
# EXP002 CELL 16
# FINAL REPORT CONSISTENCY & QUALITY-CONTROL AUDIT
# ============================================================

import json
from pathlib import Path
from datetime import datetime

print("=" * 70)
print("EXP002 FINAL REPORT CONSISTENCY & QUALITY-CONTROL AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE_DIR = Path(
    r"G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation"
)

EXP_DIR = BASE_DIR / "05_Experiments" / "Evaluation" / "Objective" / "EXP002"
REPORTS_DIR = EXP_DIR / "reports"

print("\n" + "-" * 70)
print("INPUT VALIDATION")
print("-" * 70)

if not REPORTS_DIR.exists():
    raise FileNotFoundError(f"Reports directory not found: {REPORTS_DIR}")

print(f"Reports directory: {REPORTS_DIR}")

# ------------------------------------------------------------
# EXPECTED REPORTS
# ------------------------------------------------------------

EXPECTED_REPORTS = {
    "integrity": "metrics_integrity_audit.json",
    "descriptive": "exp002_metric_descriptive_statistics.json",
    "distribution": "exp002_distribution_outlier_analysis.json",
    "source_comparison": "exp002_source_level_comparative_analysis.json",
    "track_aggregate": "exp002_track_level_aggregate_analysis.json",
    "consolidated": "exp002_consolidated_objective_analysis.json",
}

report_paths = {}

for key, filename in EXPECTED_REPORTS.items():
    path = REPORTS_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Required EXP002 report is missing: {path}"
        )

    report_paths[key] = path
    print(f"✓ {key.replace('_', ' ').title()}: {path}")

# ------------------------------------------------------------
# LOAD REPORTS
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("REPORT LOADING")
print("-" * 70)

reports = {}

for key, path in report_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        reports[key] = json.load(f)

    print(f"✓ Loaded: {path.name}")

# ------------------------------------------------------------
# EXPECTED EXP002 CONFIGURATION
# ------------------------------------------------------------

EXPECTED_EXPERIMENT = "EXP002"

EXPECTED_TRACKS = 50
EXPECTED_SOURCES_PER_TRACK = 4
EXPECTED_SOURCE_EVALUATIONS = 200

EXPECTED_SOURCES = [
    "vocals",
    "drums",
    "bass",
    "other",
]

APPROVED_METRICS = [
    "SDR",
    "SIR",
    "SAR",
    "SI-SDR",
]

EXCLUDED_METRICS = [
    "ISR",
]

METRIC_KEY_MAP = {
    "SDR": "SDR",
    "SIR": "SIR",
    "SAR": "SAR",
    "SI-SDR": "SI_SDR",
}

# ------------------------------------------------------------
# AUDIT STATE
# ------------------------------------------------------------

checks = []
errors = []
warnings = []

def check(name, condition, detail=""):
    if condition:
        checks.append(name)
        print(f"✓ {name}")
    else:
        errors.append(name)
        print(f"✗ {name}")
        if detail:
            print(f"  {detail}")

def warning(name, detail=""):
    warnings.append(name)
    print(f"⚠ {name}")
    if detail:
        print(f"  {detail}")

# ------------------------------------------------------------
# EXPERIMENT ID CHECK
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("EXPERIMENT IDENTIFICATION")
print("-" * 70)

descriptive = reports["descriptive"]
distribution = reports["distribution"]
source_comparison = reports["source_comparison"]
track_aggregate = reports["track_aggregate"]
consolidated = reports["consolidated"]

if "experiment" in descriptive:
    check(
        "Descriptive report identifies EXP002",
        descriptive["experiment"] == EXPECTED_EXPERIMENT
    )

# Some reports may not contain an explicit experiment field.
# Their filenames and directory placement already establish context.

# ------------------------------------------------------------
# METRIC CONFIGURATION
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("METRIC CONFIGURATION")
print("-" * 70)

check(
    "Approved metrics match expected set",
    descriptive.get("approved_metrics") == APPROVED_METRICS,
    f"Found: {descriptive.get('approved_metrics')}"
)

check(
    "Excluded metrics match expected set",
    descriptive.get("excluded_metrics") == EXCLUDED_METRICS,
    f"Found: {descriptive.get('excluded_metrics')}"
)

# Check consolidated report if the corresponding fields exist.

if "approved_metrics" in consolidated:
    check(
        "Consolidated approved metrics match",
        consolidated["approved_metrics"] == APPROVED_METRICS,
        f"Found: {consolidated['approved_metrics']}"
    )

if "excluded_metrics" in consolidated:
    check(
        "Consolidated excluded metrics match",
        consolidated["excluded_metrics"] == EXCLUDED_METRICS,
        f"Found: {consolidated['excluded_metrics']}"
    )

# ------------------------------------------------------------
# DATASET COUNTS
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("DATASET COUNT CONSISTENCY")
print("-" * 70)

dataset = descriptive.get("dataset", {})

check(
    "Descriptive report track count = 50",
    dataset.get("tracks") == EXPECTED_TRACKS,
    f"Found: {dataset.get('tracks')}"
)

check(
    "Descriptive report sources per track = 4",
    dataset.get("sources_per_track") == EXPECTED_SOURCES_PER_TRACK,
    f"Found: {dataset.get('sources_per_track')}"
)

check(
    "Descriptive report source evaluations = 200",
    dataset.get("source_evaluations") == EXPECTED_SOURCE_EVALUATIONS,
    f"Found: {dataset.get('source_evaluations')}"
)

# ------------------------------------------------------------
# SOURCE COVERAGE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SOURCE COVERAGE CONSISTENCY")
print("-" * 70)

source_stats = descriptive.get("source_level_statistics", {})
discovered_sources = list(source_stats.keys())

print(f"Sources discovered: {discovered_sources}")

check(
    "Source set matches expected four sources",
    set(discovered_sources) == set(EXPECTED_SOURCES),
    f"Found: {discovered_sources}"
)

for source in EXPECTED_SOURCES:
    check(
        f"{source} source statistics present",
        source in source_stats
    )

    if source in source_stats:
        for metric in APPROVED_METRICS:
            check(
                f"{source} contains {metric}",
                metric in source_stats[source]
            )

# ------------------------------------------------------------
# OVERALL SOURCE-LEVEL STATISTICS
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("OVERALL SOURCE-LEVEL STATISTICS")
print("-" * 70)

overall_stats = descriptive.get(
    "overall_source_level_statistics",
    {}
)

check(
    "All approved metrics present in overall statistics",
    set(overall_stats.keys()) == set(APPROVED_METRICS),
    f"Found: {list(overall_stats.keys())}"
)

for metric in APPROVED_METRICS:
    if metric not in overall_stats:
        continue

    stats = overall_stats[metric]

    check(
        f"{metric} source-level count = 200",
        stats.get("count") == EXPECTED_SOURCE_EVALUATIONS,
        f"Found: {stats.get('count')}"
    )

# ------------------------------------------------------------
# TRACK-AVERAGE STATISTICS
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK-AVERAGE STATISTICS")
print("-" * 70)

track_stats = descriptive.get(
    "track_average_statistics",
    {}
)

check(
    "All approved metrics present in track-average statistics",
    set(track_stats.keys()) == set(APPROVED_METRICS),
    f"Found: {list(track_stats.keys())}"
)

for metric in APPROVED_METRICS:
    if metric not in track_stats:
        continue

    stats = track_stats[metric]

    check(
        f"{metric} track-average count = 50",
        stats.get("count") == EXPECTED_TRACKS,
        f"Found: {stats.get('count')}"
    )

# ------------------------------------------------------------
# SOURCE COMPARISON CONSISTENCY
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SOURCE COMPARISON CONSISTENCY")
print("-" * 70)

# The source comparison report uses source-level statistics.
# Compare its values against the descriptive report.

source_comparison_stats = (
    source_comparison.get("source_statistics")
    or source_comparison.get("source_level_statistics")
    or source_comparison.get("sources")
)

if source_comparison_stats is None:
    warning(
        "Source comparison schema does not expose a directly comparable statistics dictionary",
        "Earlier Cell 13 output is still considered valid; no values were overwritten."
    )
else:
    for source in EXPECTED_SOURCES:
        if source not in source_comparison_stats:
            warning(
                f"{source} not directly exposed in source comparison schema"
            )
            continue

        for metric in APPROVED_METRICS:
            if metric not in source_comparison_stats[source]:
                continue

            comparison_value = source_comparison_stats[source][metric]

            # Support either a raw numeric value or a statistics dictionary.
            if isinstance(comparison_value, dict):
                comparison_mean = (
                    comparison_value.get("mean_dB")
                    or comparison_value.get("mean")
                )
            else:
                comparison_mean = comparison_value

            descriptive_value = source_stats[source][metric].get("mean_dB")

            if (
                isinstance(comparison_mean, (int, float))
                and isinstance(descriptive_value, (int, float))
            ):
                tolerance = 1e-9

                check(
                    f"{source} {metric} mean agrees between reports",
                    abs(comparison_mean - descriptive_value) <= tolerance,
                    f"Comparison={comparison_mean}, Descriptive={descriptive_value}"
                )

# ------------------------------------------------------------
# TRACK-AVERAGE MEAN CONSISTENCY
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK-AVERAGE / SOURCE-LEVEL MEAN CONSISTENCY")
print("-" * 70)

for metric in APPROVED_METRICS:

    source_mean = overall_stats[metric]["mean_dB"]
    track_mean = track_stats[metric]["mean_dB"]

    check(
        f"{metric}: source-level and track-average means agree",
        abs(source_mean - track_mean) <= 1e-9,
        f"Source-level={source_mean}, Track-average={track_mean}"
    )

# ------------------------------------------------------------
# TRACK AGGREGATE VALIDATION
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("TRACK AGGREGATE REPORT CONSISTENCY")
print("-" * 70)

# Cell 14 output establishes that the report contains 50 tracks.
# Inspect common schema locations if available.

aggregate_dataset = (
    track_aggregate.get("dataset")
    or track_aggregate.get("summary")
    or {}
)

if isinstance(aggregate_dataset, dict):

    if "tracks" in aggregate_dataset:
        check(
            "Track aggregate report track count = 50",
            aggregate_dataset["tracks"] == EXPECTED_TRACKS,
            f"Found: {aggregate_dataset['tracks']}"
        )

    if "source_evaluations" in aggregate_dataset:
        check(
            "Track aggregate report source evaluations = 200",
            aggregate_dataset["source_evaluations"]
            == EXPECTED_SOURCE_EVALUATIONS,
            f"Found: {aggregate_dataset['source_evaluations']}"
        )

# ------------------------------------------------------------
# CONSOLIDATED REPORT CHECK
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("CONSOLIDATED REPORT CONSISTENCY")
print("-" * 70)

# Check consolidated report for count fields wherever available.

consolidated_checks = {
    "tracks": EXPECTED_TRACKS,
    "source_evaluations": EXPECTED_SOURCE_EVALUATIONS,
}

for key, expected in consolidated_checks.items():

    if key in consolidated:
        check(
            f"Consolidated {key} = {expected}",
            consolidated[key] == expected,
            f"Found: {consolidated[key]}"
        )

# ------------------------------------------------------------
# NUMERICAL SANITY CHECKS
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("NUMERICAL SANITY CHECKS")
print("-" * 70)

for metric in APPROVED_METRICS:

    stats = overall_stats[metric]

    values = [
        stats.get("mean_dB"),
        stats.get("median_dB"),
        stats.get("std_dB"),
        stats.get("min_dB"),
        stats.get("q1_dB"),
        stats.get("q3_dB"),
        stats.get("max_dB"),
        stats.get("iqr_dB"),
    ]

    check(
        f"{metric} statistics are numeric",
        all(isinstance(v, (int, float)) for v in values)
    )

    if all(isinstance(v, (int, float)) for v in values):

        check(
            f"{metric}: min <= Q1 <= median <= Q3 <= max",
            (
                stats["min_dB"]
                <= stats["q1_dB"]
                <= stats["median_dB"]
                <= stats["q3_dB"]
                <= stats["max_dB"]
            )
        )

        check(
            f"{metric}: standard deviation >= 0",
            stats["std_dB"] >= 0
        )

        check(
            f"{metric}: IQR >= 0",
            stats["iqr_dB"] >= 0
        )

# ------------------------------------------------------------
# EXCLUDED METRIC CHECK
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("EXCLUDED METRIC CHECK")
print("-" * 70)

# ISR should not appear in approved analysis statistics.

for section_name, section in [
    ("overall source-level statistics", overall_stats),
    ("track-average statistics", track_stats),
]:

    check(
        f"ISR absent from {section_name}",
        "ISR" not in section
    )

# ------------------------------------------------------------
# FINAL AUDIT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 16 RESULT")
print("=" * 70)

print(f"\nChecks passed:       {len(checks)}")
print(f"Checks failed:       {len(errors)}")
print(f"Warnings:            {len(warnings)}")

print("\nCore EXP002 configuration:")
print(f"  Tracks:             {EXPECTED_TRACKS}")
print(f"  Sources per track:  {EXPECTED_SOURCES_PER_TRACK}")
print(f"  Source evaluations: {EXPECTED_SOURCE_EVALUATIONS}")
print(f"  Approved metrics:   {', '.join(APPROVED_METRICS)}")
print(f"  Excluded metric:    ISR")

if errors:
    print("\n" + "-" * 70)
    print("FAILED CHECKS")
    print("-" * 70)

    for item in errors:
        print(f"  ✗ {item}")

    raise RuntimeError(
        f"EXP002 final QC audit failed with {len(errors)} check(s). "
        "Review the failed checks above before proceeding."
    )

print("\n✓ All critical EXP002 consistency checks passed.")

if warnings:
    print("\nWarnings:")
    for item in warnings:
        print(f"  ⚠ {item}")

# ------------------------------------------------------------
# SAVE QC REPORT
# ------------------------------------------------------------

qc_report = {
    "experiment": EXPECTED_EXPERIMENT,
    "audit_type": "final_report_consistency_and_quality_control",
    "timestamp": datetime.now().isoformat(),

    "configuration": {
        "tracks": EXPECTED_TRACKS,
        "sources_per_track": EXPECTED_SOURCES_PER_TRACK,
        "source_evaluations": EXPECTED_SOURCE_EVALUATIONS,
        "sources": EXPECTED_SOURCES,
        "approved_metrics": APPROVED_METRICS,
        "excluded_metrics": EXCLUDED_METRICS,
    },

    "checks": {
        "passed": len(checks),
        "failed": len(errors),
        "warnings": len(warnings),
    },

    "status": "PASS",

    "reports_audited": {
        key: str(path)
        for key, path in report_paths.items()
    },

    "notes": [
        "Audit performed using existing EXP002 JSON reports only.",
        "No WAV files were loaded.",
        "No objective metrics were recomputed.",
        "ISR remains excluded from approved analysis.",
    ],
}

qc_path = REPORTS_DIR / "exp002_final_report_consistency_audit.json"

with open(qc_path, "w", encoding="utf-8") as f:
    json.dump(qc_report, f, indent=2)

print("\nQC report:")
print(f"  {qc_path}")

print("\n" + "=" * 70)
print("✓ CELL 16 COMPLETE")
print("=" * 70)

print(
    "\nEXP002 final report consistency and quality-control audit "
    "completed successfully."
)

EXP002 FINAL REPORT CONSISTENCY & QUALITY-CONTROL AUDIT

----------------------------------------------------------------------
INPUT VALIDATION
----------------------------------------------------------------------
Reports directory: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports
✓ Integrity: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports\metrics_integrity_audit.json
✓ Descriptive: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports\exp002_metric_descriptive_statistics.json
✓ Distribution: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports\exp002_distribution_outlier_analysis.json
✓ Source Comparison: G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports\exp002_

# Final Package & Completion Manifest

In [22]:
# ============================================================
# EXP002 FINAL PACKAGE & COMPLETION MANIFEST
# ============================================================

from pathlib import Path
import json
from datetime import datetime

print("=" * 70)
print("EXP002 FINAL PACKAGE & COMPLETION MANIFEST")
print("=" * 70)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE_DIR = Path(
    r"G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation"
)

EXP_DIR = BASE_DIR / "05_Experiments" / "Evaluation" / "Objective" / "EXP002"
METRICS_DIR = EXP_DIR / "metrics"
REPORTS_DIR = EXP_DIR / "reports"
PLOTS_DIR = EXP_DIR / "plots"

# ------------------------------------------------------------
# EXPECTED FILES
# ------------------------------------------------------------

expected_metrics = [
    METRICS_DIR / "per_track_metrics.csv",
    METRICS_DIR / "per_track_metrics.jsonl",
]

expected_reports = [
    REPORTS_DIR / "metrics_integrity_audit.json",
    REPORTS_DIR / "exp002_metric_descriptive_statistics.json",
    REPORTS_DIR / "exp002_distribution_outlier_analysis.json",
    REPORTS_DIR / "exp002_source_level_comparative_analysis.json",
    REPORTS_DIR / "exp002_track_level_aggregate_analysis.json",
    REPORTS_DIR / "exp002_consolidated_objective_analysis.json",
    REPORTS_DIR / "exp002_final_report_consistency_audit.json",
]

expected_plots = [
    PLOTS_DIR / "exp002_sdr_distribution.png",
    PLOTS_DIR / "exp002_sir_distribution.png",
    PLOTS_DIR / "exp002_sar_distribution.png",
    PLOTS_DIR / "exp002_si_sdr_distribution.png",
    PLOTS_DIR / "exp002_sdr_by_source_boxplot.png",
    PLOTS_DIR / "exp002_sir_by_source_boxplot.png",
    PLOTS_DIR / "exp002_sar_by_source_boxplot.png",
    PLOTS_DIR / "exp002_si_sdr_by_source_boxplot.png",
]

# ------------------------------------------------------------
# DIRECTORY VALIDATION
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("DIRECTORY VALIDATION")
print("-" * 70)

for directory in [EXP_DIR, METRICS_DIR, REPORTS_DIR, PLOTS_DIR]:
    if not directory.exists():
        raise FileNotFoundError(
            f"Required directory does not exist: {directory}"
        )

    print(f"✓ {directory}")

# ------------------------------------------------------------
# FILE VALIDATION
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("METRIC FILE VALIDATION")
print("-" * 70)

missing_metrics = []

for path in expected_metrics:
    if path.exists():
        print(f"✓ {path.name}")
    else:
        print(f"✗ MISSING: {path.name}")
        missing_metrics.append(str(path))

print("\n" + "-" * 70)
print("REPORT FILE VALIDATION")
print("-" * 70)

missing_reports = []

for path in expected_reports:
    if path.exists():
        print(f"✓ {path.name}")
    else:
        print(f"✗ MISSING: {path.name}")
        missing_reports.append(str(path))

print("\n" + "-" * 70)
print("PLOT FILE VALIDATION")
print("-" * 70)

missing_plots = []

for path in expected_plots:
    if path.exists():
        print(f"✓ {path.name}")
    else:
        print(f"✗ MISSING: {path.name}")
        missing_plots.append(str(path))

# ------------------------------------------------------------
# LOAD QC REPORT
# ------------------------------------------------------------

qc_path = (
    REPORTS_DIR /
    "exp002_final_report_consistency_audit.json"
)

if not qc_path.exists():
    raise FileNotFoundError(
        f"Final QC report not found: {qc_path}"
    )

with open(qc_path, "r", encoding="utf-8") as f:
    qc_report = json.load(f)

# ------------------------------------------------------------
# EXTRACT QC INFORMATION
# ------------------------------------------------------------

checks_passed = qc_report.get("checks_passed", None)
checks_failed = qc_report.get("checks_failed", None)
warnings = qc_report.get("warnings", None)

# Handle possible alternate field names safely
if checks_passed is None:
    checks_passed = qc_report.get("passed", None)

if checks_failed is None:
    checks_failed = qc_report.get("failed", None)

if warnings is None:
    warnings = qc_report.get("warning_count", None)

# ------------------------------------------------------------
# FINAL CONFIGURATION
# ------------------------------------------------------------

approved_metrics = [
    "SDR",
    "SIR",
    "SAR",
    "SI-SDR",
]

excluded_metrics = [
    "ISR",
]

sources = [
    "vocals",
    "drums",
    "bass",
    "other",
]

# ------------------------------------------------------------
# COMPLETION STATUS
# ------------------------------------------------------------

all_files_present = (
    len(missing_metrics) == 0
    and len(missing_reports) == 0
    and len(missing_plots) == 0
)

qc_passed = (
    checks_failed == 0
    if checks_failed is not None
    else True
)

experiment_complete = all_files_present and qc_passed

# ------------------------------------------------------------
# COMPLETION MANIFEST
# ------------------------------------------------------------

manifest = {
    "experiment": "EXP002",
    "experiment_name": "Objective Evaluation Analysis",
    "generated_at": datetime.now().isoformat(),

    "status": (
        "COMPLETE"
        if experiment_complete
        else "INCOMPLETE"
    ),

    "dataset": {
        "tracks": 50,
        "sources_per_track": 4,
        "source_evaluations": 200,
        "sources": sources,
    },

    "metrics": {
        "approved": approved_metrics,
        "excluded": excluded_metrics,
    },

    "analysis_stages_completed": [
        "Metric integrity audit",
        "Descriptive statistics",
        "Distribution analysis",
        "IQR-based outlier analysis",
        "Source-level comparative analysis",
        "Track-level aggregate analysis",
        "Consolidated objective analysis",
        "Final report consistency and QC audit",
    ],

    "objective_evaluation_policy": {
        "metric_recomputation_in_analysis_cells": False,
        "wav_files_loaded_in_analysis_cells": False,
        "analysis_based_on_existing_metric_outputs": True,
        "excluded_metric": "ISR",
    },

    "quality_control": {
        "checks_passed": checks_passed,
        "checks_failed": checks_failed,
        "warnings": warnings,
        "qc_status": (
            "PASSED"
            if qc_passed
            else "FAILED"
        ),
    },

    "files": {
        "metric_files": [
            str(path) for path in expected_metrics
            if path.exists()
        ],
        "report_files": [
            str(path) for path in expected_reports
            if path.exists()
        ],
        "plot_files": [
            str(path) for path in expected_plots
            if path.exists()
        ],
    },

    "missing_files": {
        "metrics": missing_metrics,
        "reports": missing_reports,
        "plots": missing_plots,
    },

    "completion_statement": (
        "EXP002 objective evaluation analysis and quality-control "
        "processing completed successfully using the existing "
        "objective metric outputs."
        if experiment_complete
        else
        "EXP002 package validation is incomplete. "
        "One or more required files or QC conditions are missing."
    ),
}

# ------------------------------------------------------------
# WRITE MANIFEST
# ------------------------------------------------------------

manifest_path = (
    REPORTS_DIR /
    "exp002_completion_manifest.json"
)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 17 RESULT")
print("=" * 70)

print(f"\nExperiment status:      {manifest['status']}")
print(f"Tracks analysed:       50")
print(f"Source evaluations:    200")
print(f"Approved metrics:      {len(approved_metrics)}")
print(f"Excluded metric:       ISR")

print("\nRequired files:")
print(f"  Metric files:         {len(expected_metrics) - len(missing_metrics)}/{len(expected_metrics)}")
print(f"  Report files:         {len(expected_reports) - len(missing_reports)}/{len(expected_reports)}")
print(f"  Plot files:           {len(expected_plots) - len(missing_plots)}/{len(expected_plots)}")

print("\nQuality control:")
if checks_passed is not None:
    print(f"  Checks passed:        {checks_passed}")
if checks_failed is not None:
    print(f"  Checks failed:        {checks_failed}")
if warnings is not None:
    print(f"  Warnings:             {warnings}")

print(f"  QC status:            {manifest['quality_control']['qc_status']}")

print("\nAnalysis stages:")
for stage in manifest["analysis_stages_completed"]:
    print(f"  ✓ {stage}")

print("\nFinal package:")
print(f"  ✓ Metrics:            {METRICS_DIR}")
print(f"  ✓ Reports:            {REPORTS_DIR}")
print(f"  ✓ Plots:              {PLOTS_DIR}")

print("\nCompletion manifest:")
print(f"  {manifest_path}")

print("\n" + "=" * 70)

if experiment_complete:
    print("✓ EXP002 COMPLETE")
    print("=" * 70)
    print(
        "All required EXP002 metric outputs, reports, plots, "
        "and QC artifacts are present."
    )
    print(
        "No WAV files were loaded and no objective metrics "
        "were recomputed during the analysis stages."
    )
else:
    print("✗ EXP002 INCOMPLETE")
    print("=" * 70)

    if missing_metrics:
        print("\nMissing metric files:")
        for item in missing_metrics:
            print(f"  - {item}")

    if missing_reports:
        print("\nMissing report files:")
        for item in missing_reports:
            print(f"  - {item}")

    if missing_plots:
        print("\nMissing plot files:")
        for item in missing_plots:
            print(f"  - {item}")

print("=" * 70)

EXP002 FINAL PACKAGE & COMPLETION MANIFEST

----------------------------------------------------------------------
DIRECTORY VALIDATION
----------------------------------------------------------------------
✓ G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002
✓ G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\metrics
✓ G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\reports
✓ G:\My Drive\IKS_Music_Source_Separation_and_Instrument_Isolation\05_Experiments\Evaluation\Objective\EXP002\plots

----------------------------------------------------------------------
METRIC FILE VALIDATION
----------------------------------------------------------------------
✓ per_track_metrics.csv
✓ per_track_metrics.jsonl

----------------------------------------------------------------------
REPORT FILE VALIDATION
--------------------